# Alerta Satipo — modelo experimental con 300 lecturas
Ejecuta **Entorno de ejecución → Ejecutar todas**. CPU es suficiente. Los mismos 300 registros del repositorio están incluidos: no necesitas conectar Supabase ni proporcionar claves.

Objetivo: clasificar `fire_alarm` a partir de mediciones contemporáneas. **No es pronóstico de un incendio futuro ni un sistema validado de alertas.** Los datos son históricos externos, no mediciones de campo en Satipo.

Protocolo fijado antes de evaluar: 240 filas más antiguas para entrenamiento y 60 más recientes para prueba; un Random Forest de configuración fija y un clasificador mayoritario de referencia. No ajustar parámetros tras mirar el resultado de prueba. La ausencia de identificadores de sesión impide garantizar independencia entre ensayos; las métricas solo describen esta partición pequeña.


In [ ]:
import json, hashlib, platform, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
try:
    from IPython.display import display
except ImportError:
    display = print
OUT = Path('salida_satipo'); OUT.mkdir(exist_ok=True)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)


## 1. Cargar y verificar exactamente las 300 filas
Esta es una copia de la muestra, no una sincronización en vivo con Supabase. Se comprueba el hash, los identificadores y la distribución de etiquetas.

In [ ]:
payload = '[{"source_row":201,"utc_seconds":1654733532,"temperature_c":22.623,"humidity_pct":50.05,"tvoc_ppb":44,"eco2_ppm":444,"raw_h2":12574,"raw_ethanol":19638,"pressure_hpa":939.811,"pm1_0":0.5,"pm2_5":1.13,"nc0_5":1.89,"nc1_0":1.267,"nc2_5":0.606,"cnt":201,"fire_alarm":false,"recorded_at":"2022-06-09T00:12:12+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":457,"utc_seconds":1654733788,"temperature_c":25.039,"humidity_pct":57.22,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":12759,"raw_ethanol":19771,"pressure_hpa":939.833,"pm1_0":0.28,"pm2_5":0.32,"nc0_5":1.8,"nc1_0":0.342,"nc2_5":0.044,"cnt":457,"fire_alarm":false,"recorded_at":"2022-06-09T00:16:28+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":484,"utc_seconds":1654733815,"temperature_c":25.237,"humidity_pct":49.44,"tvoc_ppb":27,"eco2_ppm":400,"raw_h2":12759,"raw_ethanol":19763,"pressure_hpa":939.826,"pm1_0":0.18,"pm2_5":0.19,"nc0_5":1.26,"nc1_0":0.197,"nc2_5":0.004,"cnt":484,"fire_alarm":false,"recorded_at":"2022-06-09T00:16:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":892,"utc_seconds":1654734223,"temperature_c":27.005,"humidity_pct":52.58,"tvoc_ppb":27,"eco2_ppm":400,"raw_h2":12993,"raw_ethanol":19906,"pressure_hpa":939.756,"pm1_0":0.15,"pm2_5":0.15,"nc0_5":1.01,"nc1_0":0.158,"nc2_5":0.004,"cnt":892,"fire_alarm":false,"recorded_at":"2022-06-09T00:23:43+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":949,"utc_seconds":1654734280,"temperature_c":27.081,"humidity_pct":52.96,"tvoc_ppb":31,"eco2_ppm":400,"raw_h2":13006,"raw_ethanol":19907,"pressure_hpa":939.745,"pm1_0":0.21,"pm2_5":0.21,"nc0_5":1.42,"nc1_0":0.222,"nc2_5":0.005,"cnt":949,"fire_alarm":false,"recorded_at":"2022-06-09T00:24:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1018,"utc_seconds":1654734349,"temperature_c":27.122,"humidity_pct":56.25,"tvoc_ppb":11,"eco2_ppm":400,"raw_h2":13030,"raw_ethanol":19935,"pressure_hpa":939.757,"pm1_0":0.24,"pm2_5":0.25,"nc0_5":1.68,"nc1_0":0.262,"nc2_5":0.006,"cnt":1018,"fire_alarm":false,"recorded_at":"2022-06-09T00:25:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1075,"utc_seconds":1654734406,"temperature_c":27.114,"humidity_pct":56.48,"tvoc_ppb":8,"eco2_ppm":400,"raw_h2":13046,"raw_ethanol":19942,"pressure_hpa":939.805,"pm1_0":0.32,"pm2_5":0.34,"nc0_5":2.23,"nc1_0":0.348,"nc2_5":0.008,"cnt":1075,"fire_alarm":false,"recorded_at":"2022-06-09T00:26:46+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1162,"utc_seconds":1654734493,"temperature_c":27.031,"humidity_pct":49.3,"tvoc_ppb":22,"eco2_ppm":400,"raw_h2":13056,"raw_ethanol":19941,"pressure_hpa":939.783,"pm1_0":0.4,"pm2_5":0.41,"nc0_5":2.73,"nc1_0":0.425,"nc2_5":0.01,"cnt":1162,"fire_alarm":false,"recorded_at":"2022-06-09T00:28:13+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1371,"utc_seconds":1654734702,"temperature_c":26.504,"humidity_pct":50.04,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13103,"raw_ethanol":20002,"pressure_hpa":939.708,"pm1_0":0.35,"pm2_5":0.37,"nc0_5":2.43,"nc1_0":0.379,"nc2_5":0.009,"cnt":1371,"fire_alarm":false,"recorded_at":"2022-06-09T00:31:42+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1846,"utc_seconds":1654735177,"temperature_c":23.805,"humidity_pct":47.98,"tvoc_ppb":57,"eco2_ppm":400,"raw_h2":13126,"raw_ethanol":20002,"pressure_hpa":939.599,"pm1_0":0.56,"pm2_5":0.58,"nc0_5":3.87,"nc1_0":0.604,"nc2_5":0.014,"cnt":1846,"fire_alarm":false,"recorded_at":"2022-06-09T00:39:37+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1888,"utc_seconds":1654735219,"temperature_c":23.479,"humidity_pct":52.54,"tvoc_ppb":49,"eco2_ppm":400,"raw_h2":13129,"raw_ethanol":20005,"pressure_hpa":939.621,"pm1_0":0.79,"pm2_5":0.82,"nc0_5":5.42,"nc1_0":0.845,"nc2_5":0.019,"cnt":1888,"fire_alarm":false,"recorded_at":"2022-06-09T00:40:19+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":1956,"utc_seconds":1654735287,"temperature_c":22.927,"humidity_pct":48.9,"tvoc_ppb":45,"eco2_ppm":400,"raw_h2":13134,"raw_ethanol":20008,"pressure_hpa":939.576,"pm1_0":0.67,"pm2_5":0.69,"nc0_5":4.6,"nc1_0":0.717,"nc2_5":0.016,"cnt":1956,"fire_alarm":false,"recorded_at":"2022-06-09T00:41:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2015,"utc_seconds":1654735346,"temperature_c":22.423,"humidity_pct":48.69,"tvoc_ppb":50,"eco2_ppm":400,"raw_h2":13133,"raw_ethanol":20003,"pressure_hpa":939.624,"pm1_0":0.73,"pm2_5":0.76,"nc0_5":5.0,"nc1_0":0.78,"nc2_5":0.018,"cnt":2015,"fire_alarm":false,"recorded_at":"2022-06-09T00:42:26+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2038,"utc_seconds":1654735369,"temperature_c":22.22,"humidity_pct":49.96,"tvoc_ppb":68,"eco2_ppm":400,"raw_h2":13117,"raw_ethanol":19989,"pressure_hpa":939.644,"pm1_0":0.91,"pm2_5":0.94,"nc0_5":6.24,"nc1_0":0.973,"nc2_5":0.022,"cnt":2038,"fire_alarm":false,"recorded_at":"2022-06-09T00:42:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2197,"utc_seconds":1654735528,"temperature_c":20.733,"humidity_pct":48.01,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13191,"raw_ethanol":20119,"pressure_hpa":939.624,"pm1_0":0.93,"pm2_5":0.96,"nc0_5":6.37,"nc1_0":0.994,"nc2_5":0.022,"cnt":2197,"fire_alarm":false,"recorded_at":"2022-06-09T00:45:28+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2437,"utc_seconds":1654735768,"temperature_c":18.24,"humidity_pct":47.5,"tvoc_ppb":34,"eco2_ppm":400,"raw_h2":13205,"raw_ethanol":20121,"pressure_hpa":939.62,"pm1_0":1.1,"pm2_5":1.15,"nc0_5":7.59,"nc1_0":1.184,"nc2_5":0.027,"cnt":2437,"fire_alarm":false,"recorded_at":"2022-06-09T00:49:28+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2461,"utc_seconds":1654735792,"temperature_c":17.977,"humidity_pct":53.0,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13220,"raw_ethanol":20154,"pressure_hpa":939.633,"pm1_0":1.29,"pm2_5":1.34,"nc0_5":8.89,"nc1_0":1.387,"nc2_5":0.031,"cnt":2461,"fire_alarm":false,"recorded_at":"2022-06-09T00:49:52+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2666,"utc_seconds":1654735997,"temperature_c":15.641,"humidity_pct":47.36,"tvoc_ppb":82,"eco2_ppm":400,"raw_h2":13198,"raw_ethanol":20107,"pressure_hpa":939.655,"pm1_0":1.01,"pm2_5":1.05,"nc0_5":6.96,"nc1_0":1.085,"nc2_5":0.024,"cnt":2666,"fire_alarm":false,"recorded_at":"2022-06-09T00:53:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":2734,"utc_seconds":1654736065,"temperature_c":14.838,"humidity_pct":46.13,"tvoc_ppb":88,"eco2_ppm":400,"raw_h2":13187,"raw_ethanol":20075,"pressure_hpa":939.623,"pm1_0":0.99,"pm2_5":1.03,"nc0_5":6.8,"nc1_0":1.061,"nc2_5":0.024,"cnt":2734,"fire_alarm":false,"recorded_at":"2022-06-09T00:54:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":3113,"utc_seconds":1654736444,"temperature_c":10.193,"humidity_pct":51.24,"tvoc_ppb":171,"eco2_ppm":400,"raw_h2":13148,"raw_ethanol":19998,"pressure_hpa":939.664,"pm1_0":0.74,"pm2_5":0.77,"nc0_5":5.12,"nc1_0":0.798,"nc2_5":0.018,"cnt":3113,"fire_alarm":false,"recorded_at":"2022-06-09T01:00:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":3871,"utc_seconds":1654737202,"temperature_c":0.973,"humidity_pct":46.43,"tvoc_ppb":168,"eco2_ppm":400,"raw_h2":13214,"raw_ethanol":20045,"pressure_hpa":939.668,"pm1_0":2.02,"pm2_5":2.1,"nc0_5":13.9,"nc1_0":2.167,"nc2_5":0.049,"cnt":3871,"fire_alarm":true,"recorded_at":"2022-06-09T01:13:22+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4439,"utc_seconds":1654737770,"temperature_c":-4.742,"humidity_pct":54.19,"tvoc_ppb":90,"eco2_ppm":400,"raw_h2":13230,"raw_ethanol":20126,"pressure_hpa":939.656,"pm1_0":2.39,"pm2_5":2.49,"nc0_5":16.48,"nc1_0":2.57,"nc2_5":0.058,"cnt":4439,"fire_alarm":true,"recorded_at":"2022-06-09T01:22:50+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4441,"utc_seconds":1654737772,"temperature_c":-4.759,"humidity_pct":54.27,"tvoc_ppb":85,"eco2_ppm":400,"raw_h2":13236,"raw_ethanol":20126,"pressure_hpa":939.667,"pm1_0":2.33,"pm2_5":2.42,"nc0_5":16.04,"nc1_0":2.501,"nc2_5":0.056,"cnt":4441,"fire_alarm":true,"recorded_at":"2022-06-09T01:22:52+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4484,"utc_seconds":1654737815,"temperature_c":-5.117,"humidity_pct":52.7,"tvoc_ppb":57,"eco2_ppm":400,"raw_h2":13251,"raw_ethanol":20153,"pressure_hpa":939.652,"pm1_0":1.93,"pm2_5":2.01,"nc0_5":13.31,"nc1_0":2.076,"nc2_5":0.047,"cnt":4484,"fire_alarm":true,"recorded_at":"2022-06-09T01:23:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4871,"utc_seconds":1654738202,"temperature_c":-7.734,"humidity_pct":50.17,"tvoc_ppb":23,"eco2_ppm":400,"raw_h2":13256,"raw_ethanol":20198,"pressure_hpa":939.696,"pm1_0":2.36,"pm2_5":2.45,"nc0_5":16.25,"nc1_0":2.533,"nc2_5":0.057,"cnt":4871,"fire_alarm":true,"recorded_at":"2022-06-09T01:30:02+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4948,"utc_seconds":1654738279,"temperature_c":-8.107,"humidity_pct":48.34,"tvoc_ppb":44,"eco2_ppm":400,"raw_h2":13243,"raw_ethanol":20178,"pressure_hpa":939.691,"pm1_0":2.19,"pm2_5":2.27,"nc0_5":15.05,"nc1_0":2.347,"nc2_5":0.053,"cnt":4948,"fire_alarm":true,"recorded_at":"2022-06-09T01:31:19+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4992,"utc_seconds":1654738323,"temperature_c":-8.296,"humidity_pct":49.76,"tvoc_ppb":44,"eco2_ppm":400,"raw_h2":13241,"raw_ethanol":20170,"pressure_hpa":939.686,"pm1_0":2.08,"pm2_5":2.16,"nc0_5":14.28,"nc1_0":2.227,"nc2_5":0.05,"cnt":4992,"fire_alarm":true,"recorded_at":"2022-06-09T01:32:03+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":4999,"utc_seconds":1654738330,"temperature_c":-8.325,"humidity_pct":48.73,"tvoc_ppb":48,"eco2_ppm":400,"raw_h2":13236,"raw_ethanol":20177,"pressure_hpa":939.685,"pm1_0":2.47,"pm2_5":2.57,"nc0_5":17.03,"nc1_0":2.655,"nc2_5":0.06,"cnt":4999,"fire_alarm":true,"recorded_at":"2022-06-09T01:32:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":5263,"utc_seconds":1654738594,"temperature_c":-9.075,"humidity_pct":53.6,"tvoc_ppb":10,"eco2_ppm":400,"raw_h2":13259,"raw_ethanol":20215,"pressure_hpa":939.698,"pm1_0":2.91,"pm2_5":3.03,"nc0_5":20.04,"nc1_0":3.125,"nc2_5":0.071,"cnt":5263,"fire_alarm":true,"recorded_at":"2022-06-09T01:36:34+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":5337,"utc_seconds":1654738668,"temperature_c":-9.181,"humidity_pct":55.0,"tvoc_ppb":17,"eco2_ppm":400,"raw_h2":13259,"raw_ethanol":20216,"pressure_hpa":939.707,"pm1_0":2.28,"pm2_5":2.37,"nc0_5":15.72,"nc1_0":2.452,"nc2_5":0.055,"cnt":5337,"fire_alarm":true,"recorded_at":"2022-06-09T01:37:48+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":5533,"utc_seconds":1654738864,"temperature_c":-9.273,"humidity_pct":49.85,"tvoc_ppb":12,"eco2_ppm":400,"raw_h2":13264,"raw_ethanol":20250,"pressure_hpa":939.747,"pm1_0":2.61,"pm2_5":2.71,"nc0_5":17.94,"nc1_0":2.798,"nc2_5":0.063,"cnt":5533,"fire_alarm":true,"recorded_at":"2022-06-09T01:41:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":5873,"utc_seconds":1654739204,"temperature_c":-8.883,"humidity_pct":55.78,"tvoc_ppb":36,"eco2_ppm":400,"raw_h2":13246,"raw_ethanol":20233,"pressure_hpa":939.637,"pm1_0":2.31,"pm2_5":2.39,"nc0_5":15.87,"nc1_0":2.474,"nc2_5":0.056,"cnt":5873,"fire_alarm":true,"recorded_at":"2022-06-09T01:46:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":5974,"utc_seconds":1654739305,"temperature_c":-8.661,"humidity_pct":51.64,"tvoc_ppb":131,"eco2_ppm":400,"raw_h2":13198,"raw_ethanol":20142,"pressure_hpa":939.695,"pm1_0":2.01,"pm2_5":2.09,"nc0_5":13.83,"nc1_0":2.157,"nc2_5":0.049,"cnt":5974,"fire_alarm":true,"recorded_at":"2022-06-09T01:48:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":6484,"utc_seconds":1654739815,"temperature_c":-7.091,"humidity_pct":52.57,"tvoc_ppb":175,"eco2_ppm":400,"raw_h2":13164,"raw_ethanol":20079,"pressure_hpa":939.647,"pm1_0":0.71,"pm2_5":0.74,"nc0_5":4.9,"nc1_0":0.764,"nc2_5":0.017,"cnt":6484,"fire_alarm":true,"recorded_at":"2022-06-09T01:56:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":6849,"utc_seconds":1654740180,"temperature_c":-5.814,"humidity_pct":51.26,"tvoc_ppb":214,"eco2_ppm":400,"raw_h2":13147,"raw_ethanol":20045,"pressure_hpa":939.599,"pm1_0":0.46,"pm2_5":0.48,"nc0_5":3.19,"nc1_0":0.497,"nc2_5":0.011,"cnt":6849,"fire_alarm":true,"recorded_at":"2022-06-09T02:03:00+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7166,"utc_seconds":1654740497,"temperature_c":-4.855,"humidity_pct":49.62,"tvoc_ppb":250,"eco2_ppm":400,"raw_h2":13133,"raw_ethanol":20021,"pressure_hpa":939.522,"pm1_0":0.46,"pm2_5":0.48,"nc0_5":3.17,"nc1_0":0.494,"nc2_5":0.011,"cnt":7166,"fire_alarm":true,"recorded_at":"2022-06-09T02:08:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7518,"utc_seconds":1654740849,"temperature_c":-4.194,"humidity_pct":52.77,"tvoc_ppb":279,"eco2_ppm":400,"raw_h2":13117,"raw_ethanol":19978,"pressure_hpa":939.484,"pm1_0":0.57,"pm2_5":0.59,"nc0_5":3.93,"nc1_0":0.613,"nc2_5":0.014,"cnt":7518,"fire_alarm":true,"recorded_at":"2022-06-09T02:14:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7553,"utc_seconds":1654740884,"temperature_c":-4.161,"humidity_pct":51.39,"tvoc_ppb":267,"eco2_ppm":400,"raw_h2":13131,"raw_ethanol":19999,"pressure_hpa":939.475,"pm1_0":0.41,"pm2_5":0.42,"nc0_5":2.81,"nc1_0":0.439,"nc2_5":0.01,"cnt":7553,"fire_alarm":true,"recorded_at":"2022-06-09T02:14:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7581,"utc_seconds":1654740912,"temperature_c":-4.138,"humidity_pct":55.25,"tvoc_ppb":281,"eco2_ppm":400,"raw_h2":13119,"raw_ethanol":19984,"pressure_hpa":939.47,"pm1_0":0.28,"pm2_5":0.29,"nc0_5":1.92,"nc1_0":0.299,"nc2_5":0.007,"cnt":7581,"fire_alarm":true,"recorded_at":"2022-06-09T02:15:12+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7673,"utc_seconds":1654741004,"temperature_c":-4.091,"humidity_pct":49.9,"tvoc_ppb":279,"eco2_ppm":400,"raw_h2":13120,"raw_ethanol":19986,"pressure_hpa":939.436,"pm1_0":0.45,"pm2_5":0.47,"nc0_5":3.12,"nc1_0":0.486,"nc2_5":0.011,"cnt":7673,"fire_alarm":true,"recorded_at":"2022-06-09T02:16:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7682,"utc_seconds":1654741013,"temperature_c":-4.089,"humidity_pct":53.16,"tvoc_ppb":296,"eco2_ppm":400,"raw_h2":13112,"raw_ethanol":19963,"pressure_hpa":939.451,"pm1_0":0.4,"pm2_5":0.42,"nc0_5":2.78,"nc1_0":0.433,"nc2_5":0.01,"cnt":7682,"fire_alarm":true,"recorded_at":"2022-06-09T02:16:53+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":7711,"utc_seconds":1654741042,"temperature_c":-4.083,"humidity_pct":52.14,"tvoc_ppb":284,"eco2_ppm":400,"raw_h2":13124,"raw_ethanol":19991,"pressure_hpa":939.456,"pm1_0":0.39,"pm2_5":0.41,"nc0_5":2.69,"nc1_0":0.419,"nc2_5":0.009,"cnt":7711,"fire_alarm":true,"recorded_at":"2022-06-09T02:17:22+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8130,"utc_seconds":1654741461,"temperature_c":-4.31,"humidity_pct":55.43,"tvoc_ppb":315,"eco2_ppm":400,"raw_h2":13107,"raw_ethanol":19939,"pressure_hpa":939.375,"pm1_0":0.31,"pm2_5":0.32,"nc0_5":2.11,"nc1_0":0.329,"nc2_5":0.007,"cnt":8130,"fire_alarm":true,"recorded_at":"2022-06-09T02:24:21+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8138,"utc_seconds":1654741469,"temperature_c":-4.318,"humidity_pct":56.41,"tvoc_ppb":317,"eco2_ppm":400,"raw_h2":13102,"raw_ethanol":19946,"pressure_hpa":939.394,"pm1_0":0.19,"pm2_5":0.2,"nc0_5":1.31,"nc1_0":0.204,"nc2_5":0.005,"cnt":8138,"fire_alarm":true,"recorded_at":"2022-06-09T02:24:29+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8385,"utc_seconds":1654741716,"temperature_c":-4.618,"humidity_pct":51.92,"tvoc_ppb":321,"eco2_ppm":400,"raw_h2":13108,"raw_ethanol":19942,"pressure_hpa":939.358,"pm1_0":0.23,"pm2_5":0.24,"nc0_5":1.6,"nc1_0":0.25,"nc2_5":0.006,"cnt":8385,"fire_alarm":true,"recorded_at":"2022-06-09T02:28:36+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8450,"utc_seconds":1654741781,"temperature_c":-4.703,"humidity_pct":50.6,"tvoc_ppb":313,"eco2_ppm":400,"raw_h2":13099,"raw_ethanol":19940,"pressure_hpa":939.338,"pm1_0":0.28,"pm2_5":0.29,"nc0_5":1.91,"nc1_0":0.297,"nc2_5":0.007,"cnt":8450,"fire_alarm":true,"recorded_at":"2022-06-09T02:29:41+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8479,"utc_seconds":1654741810,"temperature_c":-4.74,"humidity_pct":56.67,"tvoc_ppb":334,"eco2_ppm":400,"raw_h2":13098,"raw_ethanol":19931,"pressure_hpa":939.325,"pm1_0":0.21,"pm2_5":0.22,"nc0_5":1.45,"nc1_0":0.227,"nc2_5":0.005,"cnt":8479,"fire_alarm":true,"recorded_at":"2022-06-09T02:30:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8839,"utc_seconds":1654742170,"temperature_c":-5.137,"humidity_pct":50.92,"tvoc_ppb":348,"eco2_ppm":400,"raw_h2":13074,"raw_ethanol":19917,"pressure_hpa":939.27,"pm1_0":0.25,"pm2_5":0.26,"nc0_5":1.72,"nc1_0":0.269,"nc2_5":0.006,"cnt":8839,"fire_alarm":true,"recorded_at":"2022-06-09T02:36:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":8876,"utc_seconds":1654742207,"temperature_c":-5.164,"humidity_pct":57.85,"tvoc_ppb":333,"eco2_ppm":400,"raw_h2":13078,"raw_ethanol":19921,"pressure_hpa":939.277,"pm1_0":0.55,"pm2_5":0.57,"nc0_5":3.77,"nc1_0":0.587,"nc2_5":0.013,"cnt":8876,"fire_alarm":true,"recorded_at":"2022-06-09T02:36:47+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":9233,"utc_seconds":1654742564,"temperature_c":-5.198,"humidity_pct":61.62,"tvoc_ppb":544,"eco2_ppm":645,"raw_h2":12816,"raw_ethanol":19715,"pressure_hpa":939.243,"pm1_0":2.62,"pm2_5":2.72,"nc0_5":18.05,"nc1_0":2.815,"nc2_5":0.064,"cnt":9233,"fire_alarm":true,"recorded_at":"2022-06-09T02:42:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":9363,"utc_seconds":1654742694,"temperature_c":-5.074,"humidity_pct":64.44,"tvoc_ppb":622,"eco2_ppm":554,"raw_h2":12866,"raw_ethanol":19637,"pressure_hpa":939.229,"pm1_0":2.29,"pm2_5":2.38,"nc0_5":15.77,"nc1_0":2.459,"nc2_5":0.056,"cnt":9363,"fire_alarm":true,"recorded_at":"2022-06-09T02:44:54+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":9498,"utc_seconds":1654742829,"temperature_c":-4.848,"humidity_pct":64.25,"tvoc_ppb":647,"eco2_ppm":682,"raw_h2":12788,"raw_ethanol":19607,"pressure_hpa":939.148,"pm1_0":2.41,"pm2_5":2.5,"nc0_5":16.56,"nc1_0":2.582,"nc2_5":0.058,"cnt":9498,"fire_alarm":true,"recorded_at":"2022-06-09T02:47:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":9846,"utc_seconds":1654743177,"temperature_c":-3.705,"humidity_pct":57.34,"tvoc_ppb":795,"eco2_ppm":943,"raw_h2":12653,"raw_ethanol":19528,"pressure_hpa":939.132,"pm1_0":2.25,"pm2_5":2.34,"nc0_5":15.5,"nc1_0":2.418,"nc2_5":0.055,"cnt":9846,"fire_alarm":true,"recorded_at":"2022-06-09T02:52:57+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":10087,"utc_seconds":1654743418,"temperature_c":-2.342,"humidity_pct":56.78,"tvoc_ppb":858,"eco2_ppm":724,"raw_h2":12772,"raw_ethanol":19507,"pressure_hpa":939.165,"pm1_0":2.24,"pm2_5":2.33,"nc0_5":15.43,"nc1_0":2.406,"nc2_5":0.054,"cnt":10087,"fire_alarm":true,"recorded_at":"2022-06-09T02:56:58+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":10325,"utc_seconds":1654743656,"temperature_c":-0.522,"humidity_pct":55.69,"tvoc_ppb":898,"eco2_ppm":689,"raw_h2":12786,"raw_ethanol":19508,"pressure_hpa":939.091,"pm1_0":2.57,"pm2_5":2.67,"nc0_5":17.67,"nc1_0":2.755,"nc2_5":0.062,"cnt":10325,"fire_alarm":true,"recorded_at":"2022-06-09T03:00:56+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":10370,"utc_seconds":1654743701,"temperature_c":-0.133,"humidity_pct":51.12,"tvoc_ppb":954,"eco2_ppm":705,"raw_h2":12776,"raw_ethanol":19490,"pressure_hpa":939.093,"pm1_0":2.61,"pm2_5":2.71,"nc0_5":17.97,"nc1_0":2.802,"nc2_5":0.063,"cnt":10370,"fire_alarm":true,"recorded_at":"2022-06-09T03:01:41+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":10737,"utc_seconds":1654744068,"temperature_c":3.446,"humidity_pct":53.39,"tvoc_ppb":1000,"eco2_ppm":665,"raw_h2":12785,"raw_ethanol":19481,"pressure_hpa":939.1,"pm1_0":2.36,"pm2_5":2.46,"nc0_5":16.27,"nc1_0":2.538,"nc2_5":0.057,"cnt":10737,"fire_alarm":true,"recorded_at":"2022-06-09T03:07:48+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":10820,"utc_seconds":1654744151,"temperature_c":4.331,"humidity_pct":52.33,"tvoc_ppb":942,"eco2_ppm":646,"raw_h2":12799,"raw_ethanol":19496,"pressure_hpa":939.064,"pm1_0":2.14,"pm2_5":2.22,"nc0_5":14.71,"nc1_0":2.294,"nc2_5":0.052,"cnt":10820,"fire_alarm":true,"recorded_at":"2022-06-09T03:09:11+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":10985,"utc_seconds":1654744316,"temperature_c":6.141,"humidity_pct":53.82,"tvoc_ppb":1053,"eco2_ppm":671,"raw_h2":12788,"raw_ethanol":19461,"pressure_hpa":939.082,"pm1_0":2.08,"pm2_5":2.16,"nc0_5":14.29,"nc1_0":2.229,"nc2_5":0.05,"cnt":10985,"fire_alarm":true,"recorded_at":"2022-06-09T03:11:56+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":11307,"utc_seconds":1654744638,"temperature_c":9.746,"humidity_pct":48.0,"tvoc_ppb":979,"eco2_ppm":629,"raw_h2":12805,"raw_ethanol":19488,"pressure_hpa":939.077,"pm1_0":1.89,"pm2_5":1.97,"nc0_5":13.02,"nc1_0":2.03,"nc2_5":0.046,"cnt":11307,"fire_alarm":true,"recorded_at":"2022-06-09T03:17:18+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12029,"utc_seconds":1654745360,"temperature_c":16.991,"humidity_pct":47.04,"tvoc_ppb":1042,"eco2_ppm":586,"raw_h2":12823,"raw_ethanol":19462,"pressure_hpa":939.011,"pm1_0":1.99,"pm2_5":2.07,"nc0_5":13.7,"nc1_0":2.136,"nc2_5":0.048,"cnt":12029,"fire_alarm":true,"recorded_at":"2022-06-09T03:29:20+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12200,"utc_seconds":1654745531,"temperature_c":18.292,"humidity_pct":47.32,"tvoc_ppb":1067,"eco2_ppm":573,"raw_h2":12827,"raw_ethanol":19469,"pressure_hpa":939.005,"pm1_0":2.64,"pm2_5":2.74,"nc0_5":18.17,"nc1_0":2.833,"nc2_5":0.064,"cnt":12200,"fire_alarm":true,"recorded_at":"2022-06-09T03:32:11+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12392,"utc_seconds":1654745723,"temperature_c":19.465,"humidity_pct":47.58,"tvoc_ppb":1208,"eco2_ppm":605,"raw_h2":12806,"raw_ethanol":19421,"pressure_hpa":938.982,"pm1_0":1.98,"pm2_5":2.06,"nc0_5":13.64,"nc1_0":2.126,"nc2_5":0.048,"cnt":12392,"fire_alarm":true,"recorded_at":"2022-06-09T03:35:23+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12416,"utc_seconds":1654745747,"temperature_c":19.587,"humidity_pct":53.21,"tvoc_ppb":1065,"eco2_ppm":570,"raw_h2":12831,"raw_ethanol":19469,"pressure_hpa":938.993,"pm1_0":2.34,"pm2_5":2.43,"nc0_5":16.07,"nc1_0":2.506,"nc2_5":0.057,"cnt":12416,"fire_alarm":true,"recorded_at":"2022-06-09T03:35:47+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12469,"utc_seconds":1654745800,"temperature_c":19.838,"humidity_pct":52.74,"tvoc_ppb":1117,"eco2_ppm":571,"raw_h2":12827,"raw_ethanol":19455,"pressure_hpa":938.998,"pm1_0":2.35,"pm2_5":2.44,"nc0_5":16.14,"nc1_0":2.517,"nc2_5":0.057,"cnt":12469,"fire_alarm":true,"recorded_at":"2022-06-09T03:36:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12501,"utc_seconds":1654745832,"temperature_c":19.975,"humidity_pct":47.17,"tvoc_ppb":1086,"eco2_ppm":574,"raw_h2":12829,"raw_ethanol":19460,"pressure_hpa":938.974,"pm1_0":1.9,"pm2_5":1.98,"nc0_5":13.1,"nc1_0":2.042,"nc2_5":0.046,"cnt":12501,"fire_alarm":true,"recorded_at":"2022-06-09T03:37:12+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":12736,"utc_seconds":1654746067,"temperature_c":20.664,"humidity_pct":47.63,"tvoc_ppb":1082,"eco2_ppm":550,"raw_h2":12836,"raw_ethanol":19461,"pressure_hpa":938.954,"pm1_0":2.04,"pm2_5":2.12,"nc0_5":14.01,"nc1_0":2.185,"nc2_5":0.049,"cnt":12736,"fire_alarm":true,"recorded_at":"2022-06-09T03:41:07+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":13188,"utc_seconds":1654746519,"temperature_c":20.644,"humidity_pct":52.63,"tvoc_ppb":1141,"eco2_ppm":534,"raw_h2":12843,"raw_ethanol":19453,"pressure_hpa":938.889,"pm1_0":1.83,"pm2_5":1.9,"nc0_5":12.62,"nc1_0":1.968,"nc2_5":0.044,"cnt":13188,"fire_alarm":true,"recorded_at":"2022-06-09T03:48:39+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":13218,"utc_seconds":1654746549,"temperature_c":20.59,"humidity_pct":47.42,"tvoc_ppb":1172,"eco2_ppm":542,"raw_h2":12840,"raw_ethanol":19452,"pressure_hpa":938.878,"pm1_0":2.13,"pm2_5":2.21,"nc0_5":14.64,"nc1_0":2.282,"nc2_5":0.052,"cnt":13218,"fire_alarm":true,"recorded_at":"2022-06-09T03:49:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":13603,"utc_seconds":1654746934,"temperature_c":19.469,"humidity_pct":52.22,"tvoc_ppb":1159,"eco2_ppm":521,"raw_h2":12841,"raw_ethanol":19446,"pressure_hpa":938.872,"pm1_0":1.8,"pm2_5":1.87,"nc0_5":12.4,"nc1_0":1.933,"nc2_5":0.044,"cnt":13603,"fire_alarm":true,"recorded_at":"2022-06-09T03:55:34+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":13941,"utc_seconds":1654747272,"temperature_c":18.006,"humidity_pct":47.27,"tvoc_ppb":1073,"eco2_ppm":479,"raw_h2":12855,"raw_ethanol":19455,"pressure_hpa":938.939,"pm1_0":2.01,"pm2_5":2.08,"nc0_5":13.8,"nc1_0":2.153,"nc2_5":0.049,"cnt":13941,"fire_alarm":true,"recorded_at":"2022-06-09T04:01:12+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":14193,"utc_seconds":1654747524,"temperature_c":16.77,"humidity_pct":48.52,"tvoc_ppb":1145,"eco2_ppm":495,"raw_h2":12846,"raw_ethanol":19450,"pressure_hpa":938.901,"pm1_0":1.62,"pm2_5":1.68,"nc0_5":11.12,"nc1_0":1.734,"nc2_5":0.039,"cnt":14193,"fire_alarm":true,"recorded_at":"2022-06-09T04:05:24+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":14333,"utc_seconds":1654747664,"temperature_c":16.071,"humidity_pct":53.14,"tvoc_ppb":1145,"eco2_ppm":483,"raw_h2":12855,"raw_ethanol":19447,"pressure_hpa":938.905,"pm1_0":2.02,"pm2_5":2.1,"nc0_5":13.91,"nc1_0":2.169,"nc2_5":0.049,"cnt":14333,"fire_alarm":true,"recorded_at":"2022-06-09T04:07:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":14507,"utc_seconds":1654747838,"temperature_c":15.22,"humidity_pct":52.67,"tvoc_ppb":1143,"eco2_ppm":470,"raw_h2":12865,"raw_ethanol":19452,"pressure_hpa":938.904,"pm1_0":1.83,"pm2_5":1.9,"nc0_5":12.62,"nc1_0":1.967,"nc2_5":0.044,"cnt":14507,"fire_alarm":true,"recorded_at":"2022-06-09T04:10:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":14812,"utc_seconds":1654748143,"temperature_c":13.872,"humidity_pct":53.16,"tvoc_ppb":1107,"eco2_ppm":452,"raw_h2":12867,"raw_ethanol":19456,"pressure_hpa":938.858,"pm1_0":1.55,"pm2_5":1.61,"nc0_5":10.69,"nc1_0":1.667,"nc2_5":0.038,"cnt":14812,"fire_alarm":true,"recorded_at":"2022-06-09T04:15:43+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":14900,"utc_seconds":1654748231,"temperature_c":13.536,"humidity_pct":47.47,"tvoc_ppb":1080,"eco2_ppm":454,"raw_h2":12863,"raw_ethanol":19455,"pressure_hpa":938.812,"pm1_0":1.77,"pm2_5":1.84,"nc0_5":12.18,"nc1_0":1.899,"nc2_5":0.043,"cnt":14900,"fire_alarm":true,"recorded_at":"2022-06-09T04:17:11+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":15926,"utc_seconds":1654749257,"temperature_c":12.035,"humidity_pct":47.34,"tvoc_ppb":1145,"eco2_ppm":410,"raw_h2":12879,"raw_ethanol":19448,"pressure_hpa":938.76,"pm1_0":1.71,"pm2_5":1.78,"nc0_5":11.79,"nc1_0":1.838,"nc2_5":0.042,"cnt":15926,"fire_alarm":true,"recorded_at":"2022-06-09T04:34:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":16225,"utc_seconds":1654749556,"temperature_c":12.333,"humidity_pct":47.39,"tvoc_ppb":1141,"eco2_ppm":422,"raw_h2":12881,"raw_ethanol":19448,"pressure_hpa":938.761,"pm1_0":2.1,"pm2_5":2.18,"nc0_5":14.45,"nc1_0":2.253,"nc2_5":0.051,"cnt":16225,"fire_alarm":true,"recorded_at":"2022-06-09T04:39:16+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":16786,"utc_seconds":1654750117,"temperature_c":13.582,"humidity_pct":52.97,"tvoc_ppb":1195,"eco2_ppm":420,"raw_h2":12891,"raw_ethanol":19452,"pressure_hpa":938.76,"pm1_0":1.78,"pm2_5":1.85,"nc0_5":12.25,"nc1_0":1.911,"nc2_5":0.043,"cnt":16786,"fire_alarm":true,"recorded_at":"2022-06-09T04:48:37+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17016,"utc_seconds":1654750347,"temperature_c":14.309,"humidity_pct":50.81,"tvoc_ppb":1124,"eco2_ppm":400,"raw_h2":12893,"raw_ethanol":19442,"pressure_hpa":938.795,"pm1_0":2.02,"pm2_5":2.1,"nc0_5":13.9,"nc1_0":2.167,"nc2_5":0.049,"cnt":17016,"fire_alarm":true,"recorded_at":"2022-06-09T04:52:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17204,"utc_seconds":1654750535,"temperature_c":14.979,"humidity_pct":52.74,"tvoc_ppb":1172,"eco2_ppm":407,"raw_h2":12891,"raw_ethanol":19427,"pressure_hpa":938.768,"pm1_0":1.46,"pm2_5":1.51,"nc0_5":10.03,"nc1_0":1.563,"nc2_5":0.035,"cnt":17204,"fire_alarm":true,"recorded_at":"2022-06-09T04:55:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17268,"utc_seconds":1654750599,"temperature_c":15.221,"humidity_pct":53.14,"tvoc_ppb":1164,"eco2_ppm":401,"raw_h2":12898,"raw_ethanol":19440,"pressure_hpa":938.8,"pm1_0":1.46,"pm2_5":1.52,"nc0_5":10.08,"nc1_0":1.572,"nc2_5":0.035,"cnt":17268,"fire_alarm":true,"recorded_at":"2022-06-09T04:56:39+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17417,"utc_seconds":1654750748,"temperature_c":15.808,"humidity_pct":47.9,"tvoc_ppb":1153,"eco2_ppm":407,"raw_h2":12905,"raw_ethanol":19443,"pressure_hpa":938.756,"pm1_0":1.44,"pm2_5":1.49,"nc0_5":9.9,"nc1_0":1.543,"nc2_5":0.035,"cnt":17417,"fire_alarm":true,"recorded_at":"2022-06-09T04:59:08+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17418,"utc_seconds":1654750749,"temperature_c":15.812,"humidity_pct":47.88,"tvoc_ppb":1132,"eco2_ppm":400,"raw_h2":12910,"raw_ethanol":19447,"pressure_hpa":938.75,"pm1_0":1.47,"pm2_5":1.53,"nc0_5":10.15,"nc1_0":1.582,"nc2_5":0.036,"cnt":17418,"fire_alarm":true,"recorded_at":"2022-06-09T04:59:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17698,"utc_seconds":1654751029,"temperature_c":16.988,"humidity_pct":53.81,"tvoc_ppb":1134,"eco2_ppm":402,"raw_h2":12905,"raw_ethanol":19440,"pressure_hpa":938.718,"pm1_0":1.51,"pm2_5":1.56,"nc0_5":10.36,"nc1_0":1.615,"nc2_5":0.036,"cnt":17698,"fire_alarm":true,"recorded_at":"2022-06-09T05:03:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17704,"utc_seconds":1654751035,"temperature_c":17.014,"humidity_pct":51.38,"tvoc_ppb":1090,"eco2_ppm":400,"raw_h2":12919,"raw_ethanol":19457,"pressure_hpa":938.712,"pm1_0":1.8,"pm2_5":1.87,"nc0_5":12.4,"nc1_0":1.933,"nc2_5":0.044,"cnt":17704,"fire_alarm":true,"recorded_at":"2022-06-09T05:03:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":17947,"utc_seconds":1654751278,"temperature_c":18.05,"humidity_pct":49.71,"tvoc_ppb":1124,"eco2_ppm":400,"raw_h2":12912,"raw_ethanol":19450,"pressure_hpa":938.699,"pm1_0":1.55,"pm2_5":1.61,"nc0_5":10.7,"nc1_0":1.668,"nc2_5":0.038,"cnt":17947,"fire_alarm":true,"recorded_at":"2022-06-09T05:07:58+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18049,"utc_seconds":1654751380,"temperature_c":18.469,"humidity_pct":53.27,"tvoc_ppb":1149,"eco2_ppm":402,"raw_h2":12910,"raw_ethanol":19441,"pressure_hpa":938.692,"pm1_0":1.9,"pm2_5":1.97,"nc0_5":13.05,"nc1_0":2.036,"nc2_5":0.046,"cnt":18049,"fire_alarm":true,"recorded_at":"2022-06-09T05:09:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18093,"utc_seconds":1654751424,"temperature_c":18.645,"humidity_pct":48.89,"tvoc_ppb":1210,"eco2_ppm":419,"raw_h2":12898,"raw_ethanol":19426,"pressure_hpa":938.703,"pm1_0":1.58,"pm2_5":1.64,"nc0_5":10.9,"nc1_0":1.699,"nc2_5":0.038,"cnt":18093,"fire_alarm":true,"recorded_at":"2022-06-09T05:10:24+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18154,"utc_seconds":1654751485,"temperature_c":18.882,"humidity_pct":49.65,"tvoc_ppb":1247,"eco2_ppm":424,"raw_h2":12897,"raw_ethanol":19421,"pressure_hpa":938.697,"pm1_0":1.99,"pm2_5":2.07,"nc0_5":13.69,"nc1_0":2.134,"nc2_5":0.048,"cnt":18154,"fire_alarm":true,"recorded_at":"2022-06-09T05:11:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18326,"utc_seconds":1654751657,"temperature_c":19.503,"humidity_pct":47.45,"tvoc_ppb":1174,"eco2_ppm":412,"raw_h2":12909,"raw_ethanol":19433,"pressure_hpa":938.715,"pm1_0":1.64,"pm2_5":1.7,"nc0_5":11.29,"nc1_0":1.76,"nc2_5":0.04,"cnt":18326,"fire_alarm":true,"recorded_at":"2022-06-09T05:14:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18590,"utc_seconds":1654751921,"temperature_c":20.267,"humidity_pct":53.36,"tvoc_ppb":1224,"eco2_ppm":418,"raw_h2":12907,"raw_ethanol":19415,"pressure_hpa":938.73,"pm1_0":1.69,"pm2_5":1.76,"nc0_5":11.64,"nc1_0":1.815,"nc2_5":0.041,"cnt":18590,"fire_alarm":true,"recorded_at":"2022-06-09T05:18:41+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18644,"utc_seconds":1654751975,"temperature_c":20.388,"humidity_pct":52.8,"tvoc_ppb":1210,"eco2_ppm":413,"raw_h2":12910,"raw_ethanol":19423,"pressure_hpa":938.727,"pm1_0":1.46,"pm2_5":1.52,"nc0_5":10.06,"nc1_0":1.569,"nc2_5":0.035,"cnt":18644,"fire_alarm":true,"recorded_at":"2022-06-09T05:19:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18658,"utc_seconds":1654751989,"temperature_c":20.417,"humidity_pct":53.76,"tvoc_ppb":1197,"eco2_ppm":401,"raw_h2":12913,"raw_ethanol":19430,"pressure_hpa":938.717,"pm1_0":1.55,"pm2_5":1.62,"nc0_5":10.7,"nc1_0":1.669,"nc2_5":0.038,"cnt":18658,"fire_alarm":true,"recorded_at":"2022-06-09T05:19:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18875,"utc_seconds":1654752206,"temperature_c":20.735,"humidity_pct":50.42,"tvoc_ppb":1210,"eco2_ppm":415,"raw_h2":12906,"raw_ethanol":19424,"pressure_hpa":938.687,"pm1_0":1.46,"pm2_5":1.51,"nc0_5":10.03,"nc1_0":1.564,"nc2_5":0.035,"cnt":18875,"fire_alarm":true,"recorded_at":"2022-06-09T05:23:26+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":18985,"utc_seconds":1654752316,"temperature_c":20.792,"humidity_pct":47.52,"tvoc_ppb":1186,"eco2_ppm":400,"raw_h2":12921,"raw_ethanol":19422,"pressure_hpa":938.704,"pm1_0":1.78,"pm2_5":1.85,"nc0_5":12.25,"nc1_0":1.91,"nc2_5":0.043,"cnt":18985,"fire_alarm":true,"recorded_at":"2022-06-09T05:25:16+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":19189,"utc_seconds":1654752520,"temperature_c":20.679,"humidity_pct":53.39,"tvoc_ppb":1212,"eco2_ppm":400,"raw_h2":12916,"raw_ethanol":19425,"pressure_hpa":938.705,"pm1_0":1.48,"pm2_5":1.54,"nc0_5":10.18,"nc1_0":1.588,"nc2_5":0.036,"cnt":19189,"fire_alarm":true,"recorded_at":"2022-06-09T05:28:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":19236,"utc_seconds":1654752567,"temperature_c":20.61,"humidity_pct":50.96,"tvoc_ppb":1254,"eco2_ppm":420,"raw_h2":12914,"raw_ethanol":19417,"pressure_hpa":938.713,"pm1_0":1.79,"pm2_5":1.86,"nc0_5":12.31,"nc1_0":1.919,"nc2_5":0.043,"cnt":19236,"fire_alarm":true,"recorded_at":"2022-06-09T05:29:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":19434,"utc_seconds":1654752765,"temperature_c":20.118,"humidity_pct":53.7,"tvoc_ppb":1241,"eco2_ppm":412,"raw_h2":12922,"raw_ethanol":19417,"pressure_hpa":938.673,"pm1_0":1.61,"pm2_5":1.67,"nc0_5":11.08,"nc1_0":1.728,"nc2_5":0.039,"cnt":19434,"fire_alarm":true,"recorded_at":"2022-06-09T05:32:45+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":19840,"utc_seconds":1654753171,"temperature_c":17.965,"humidity_pct":52.38,"tvoc_ppb":1245,"eco2_ppm":415,"raw_h2":12920,"raw_ethanol":19428,"pressure_hpa":938.73,"pm1_0":1.64,"pm2_5":1.7,"nc0_5":11.26,"nc1_0":1.756,"nc2_5":0.04,"cnt":19840,"fire_alarm":true,"recorded_at":"2022-06-09T05:39:31+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":20472,"utc_seconds":1654753803,"temperature_c":11.105,"humidity_pct":48.12,"tvoc_ppb":1130,"eco2_ppm":400,"raw_h2":12975,"raw_ethanol":19442,"pressure_hpa":938.74,"pm1_0":1.75,"pm2_5":1.82,"nc0_5":12.03,"nc1_0":1.876,"nc2_5":0.042,"cnt":20472,"fire_alarm":true,"recorded_at":"2022-06-09T05:50:03+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":20639,"utc_seconds":1654753970,"temperature_c":8.704,"humidity_pct":52.63,"tvoc_ppb":933,"eco2_ppm":400,"raw_h2":13016,"raw_ethanol":19489,"pressure_hpa":938.716,"pm1_0":2.67,"pm2_5":2.78,"nc0_5":18.4,"nc1_0":2.869,"nc2_5":0.065,"cnt":20639,"fire_alarm":true,"recorded_at":"2022-06-09T05:52:50+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":20785,"utc_seconds":1654754116,"temperature_c":6.47,"humidity_pct":47.16,"tvoc_ppb":1245,"eco2_ppm":400,"raw_h2":12965,"raw_ethanol":19417,"pressure_hpa":938.692,"pm1_0":1.89,"pm2_5":1.97,"nc0_5":13.04,"nc1_0":2.034,"nc2_5":0.046,"cnt":20785,"fire_alarm":true,"recorded_at":"2022-06-09T05:55:16+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":20820,"utc_seconds":1654754151,"temperature_c":5.919,"humidity_pct":53.64,"tvoc_ppb":1272,"eco2_ppm":400,"raw_h2":12956,"raw_ethanol":19413,"pressure_hpa":938.721,"pm1_0":1.95,"pm2_5":2.02,"nc0_5":13.4,"nc1_0":2.089,"nc2_5":0.047,"cnt":20820,"fire_alarm":true,"recorded_at":"2022-06-09T05:55:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":20843,"utc_seconds":1654754174,"temperature_c":5.554,"humidity_pct":47.41,"tvoc_ppb":1323,"eco2_ppm":402,"raw_h2":12952,"raw_ethanol":19409,"pressure_hpa":938.695,"pm1_0":1.98,"pm2_5":2.06,"nc0_5":13.65,"nc1_0":2.129,"nc2_5":0.048,"cnt":20843,"fire_alarm":true,"recorded_at":"2022-06-09T05:56:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":20914,"utc_seconds":1654754245,"temperature_c":4.416,"humidity_pct":49.64,"tvoc_ppb":1404,"eco2_ppm":437,"raw_h2":12937,"raw_ethanol":19387,"pressure_hpa":938.701,"pm1_0":1.99,"pm2_5":2.07,"nc0_5":13.71,"nc1_0":2.139,"nc2_5":0.048,"cnt":20914,"fire_alarm":true,"recorded_at":"2022-06-09T05:57:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":21453,"utc_seconds":1654754784,"temperature_c":-4.505,"humidity_pct":49.02,"tvoc_ppb":1390,"eco2_ppm":425,"raw_h2":12951,"raw_ethanol":19383,"pressure_hpa":938.703,"pm1_0":1.75,"pm2_5":1.81,"nc0_5":12.02,"nc1_0":1.874,"nc2_5":0.042,"cnt":21453,"fire_alarm":true,"recorded_at":"2022-06-09T06:06:24+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":21641,"utc_seconds":1654754972,"temperature_c":-7.562,"humidity_pct":52.56,"tvoc_ppb":1318,"eco2_ppm":410,"raw_h2":12961,"raw_ethanol":19403,"pressure_hpa":938.718,"pm1_0":2.37,"pm2_5":2.46,"nc0_5":16.3,"nc1_0":2.542,"nc2_5":0.057,"cnt":21641,"fire_alarm":true,"recorded_at":"2022-06-09T06:09:32+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":21850,"utc_seconds":1654755181,"temperature_c":-10.804,"humidity_pct":49.11,"tvoc_ppb":1281,"eco2_ppm":400,"raw_h2":12969,"raw_ethanol":19410,"pressure_hpa":938.734,"pm1_0":1.55,"pm2_5":1.61,"nc0_5":10.67,"nc1_0":1.663,"nc2_5":0.038,"cnt":21850,"fire_alarm":true,"recorded_at":"2022-06-09T06:13:01+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":22438,"utc_seconds":1654755769,"temperature_c":-18.311,"humidity_pct":54.08,"tvoc_ppb":1329,"eco2_ppm":401,"raw_h2":12964,"raw_ethanol":19395,"pressure_hpa":938.742,"pm1_0":1.7,"pm2_5":1.76,"nc0_5":11.69,"nc1_0":1.823,"nc2_5":0.041,"cnt":22438,"fire_alarm":true,"recorded_at":"2022-06-09T06:22:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":22506,"utc_seconds":1654755837,"temperature_c":-18.965,"humidity_pct":50.47,"tvoc_ppb":1333,"eco2_ppm":400,"raw_h2":12978,"raw_ethanol":19404,"pressure_hpa":938.753,"pm1_0":1.73,"pm2_5":1.79,"nc0_5":11.88,"nc1_0":1.852,"nc2_5":0.042,"cnt":22506,"fire_alarm":true,"recorded_at":"2022-06-09T06:23:57+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":22539,"utc_seconds":1654755870,"temperature_c":-19.262,"humidity_pct":52.2,"tvoc_ppb":1373,"eco2_ppm":420,"raw_h2":12961,"raw_ethanol":19384,"pressure_hpa":938.757,"pm1_0":1.93,"pm2_5":2.01,"nc0_5":13.29,"nc1_0":2.073,"nc2_5":0.047,"cnt":22539,"fire_alarm":true,"recorded_at":"2022-06-09T06:24:30+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":22614,"utc_seconds":1654755945,"temperature_c":-19.888,"humidity_pct":53.88,"tvoc_ppb":1356,"eco2_ppm":415,"raw_h2":12962,"raw_ethanol":19387,"pressure_hpa":938.753,"pm1_0":1.97,"pm2_5":2.05,"nc0_5":13.56,"nc1_0":2.114,"nc2_5":0.048,"cnt":22614,"fire_alarm":true,"recorded_at":"2022-06-09T06:25:45+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":22633,"utc_seconds":1654755964,"temperature_c":-20.035,"humidity_pct":48.66,"tvoc_ppb":1320,"eco2_ppm":400,"raw_h2":12967,"raw_ethanol":19393,"pressure_hpa":938.765,"pm1_0":1.94,"pm2_5":2.01,"nc0_5":13.33,"nc1_0":2.078,"nc2_5":0.047,"cnt":22633,"fire_alarm":true,"recorded_at":"2022-06-09T06:26:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":22789,"utc_seconds":1654756120,"temperature_c":-21.059,"humidity_pct":53.61,"tvoc_ppb":1371,"eco2_ppm":412,"raw_h2":12968,"raw_ethanol":19376,"pressure_hpa":938.737,"pm1_0":1.57,"pm2_5":1.63,"nc0_5":10.78,"nc1_0":1.68,"nc2_5":0.038,"cnt":22789,"fire_alarm":true,"recorded_at":"2022-06-09T06:28:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":23073,"utc_seconds":1654756404,"temperature_c":-21.985,"humidity_pct":49.28,"tvoc_ppb":1444,"eco2_ppm":429,"raw_h2":12960,"raw_ethanol":19364,"pressure_hpa":938.697,"pm1_0":1.79,"pm2_5":1.86,"nc0_5":12.32,"nc1_0":1.921,"nc2_5":0.043,"cnt":23073,"fire_alarm":true,"recorded_at":"2022-06-09T06:33:24+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":23105,"utc_seconds":1654756436,"temperature_c":-22.006,"humidity_pct":51.09,"tvoc_ppb":1302,"eco2_ppm":400,"raw_h2":12981,"raw_ethanol":19412,"pressure_hpa":938.724,"pm1_0":1.94,"pm2_5":2.02,"nc0_5":13.37,"nc1_0":2.085,"nc2_5":0.047,"cnt":23105,"fire_alarm":true,"recorded_at":"2022-06-09T06:33:56+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":23265,"utc_seconds":1654756596,"temperature_c":-21.835,"humidity_pct":53.29,"tvoc_ppb":1358,"eco2_ppm":406,"raw_h2":12964,"raw_ethanol":19382,"pressure_hpa":938.709,"pm1_0":1.96,"pm2_5":2.03,"nc0_5":13.47,"nc1_0":2.101,"nc2_5":0.047,"cnt":23265,"fire_alarm":true,"recorded_at":"2022-06-09T06:36:36+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":23396,"utc_seconds":1654756727,"temperature_c":-21.331,"humidity_pct":54.12,"tvoc_ppb":1366,"eco2_ppm":413,"raw_h2":12965,"raw_ethanol":19396,"pressure_hpa":938.728,"pm1_0":1.57,"pm2_5":1.63,"nc0_5":10.8,"nc1_0":1.685,"nc2_5":0.038,"cnt":23396,"fire_alarm":true,"recorded_at":"2022-06-09T06:38:47+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":23606,"utc_seconds":1654756937,"temperature_c":-19.787,"humidity_pct":47.84,"tvoc_ppb":1350,"eco2_ppm":400,"raw_h2":12979,"raw_ethanol":19390,"pressure_hpa":938.721,"pm1_0":1.63,"pm2_5":1.69,"nc0_5":11.22,"nc1_0":1.749,"nc2_5":0.04,"cnt":23606,"fire_alarm":true,"recorded_at":"2022-06-09T06:42:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":24306,"utc_seconds":1654757637,"temperature_c":-7.023,"humidity_pct":50.29,"tvoc_ppb":1270,"eco2_ppm":400,"raw_h2":13002,"raw_ethanol":19416,"pressure_hpa":938.798,"pm1_0":1.86,"pm2_5":1.93,"nc0_5":12.8,"nc1_0":1.997,"nc2_5":0.045,"cnt":24306,"fire_alarm":true,"recorded_at":"2022-06-09T06:53:57+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":24335,"utc_seconds":1654757666,"temperature_c":-6.211,"humidity_pct":50.48,"tvoc_ppb":1346,"eco2_ppm":400,"raw_h2":12989,"raw_ethanol":19395,"pressure_hpa":938.799,"pm1_0":2.05,"pm2_5":2.13,"nc0_5":14.11,"nc1_0":2.2,"nc2_5":0.05,"cnt":24335,"fire_alarm":true,"recorded_at":"2022-06-09T06:54:26+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":24358,"utc_seconds":1654757689,"temperature_c":-5.548,"humidity_pct":53.97,"tvoc_ppb":1287,"eco2_ppm":402,"raw_h2":12995,"raw_ethanol":19403,"pressure_hpa":938.791,"pm1_0":1.83,"pm2_5":1.9,"nc0_5":12.58,"nc1_0":1.961,"nc2_5":0.044,"cnt":24358,"fire_alarm":true,"recorded_at":"2022-06-09T06:54:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":24454,"utc_seconds":1654757785,"temperature_c":-2.615,"humidity_pct":50.08,"tvoc_ppb":1402,"eco2_ppm":433,"raw_h2":12979,"raw_ethanol":19378,"pressure_hpa":938.803,"pm1_0":1.99,"pm2_5":2.07,"nc0_5":13.72,"nc1_0":2.139,"nc2_5":0.048,"cnt":24454,"fire_alarm":true,"recorded_at":"2022-06-09T06:56:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":24905,"utc_seconds":1654758236,"temperature_c":15.056,"humidity_pct":51.26,"tvoc_ppb":1237,"eco2_ppm":400,"raw_h2":12999,"raw_ethanol":19414,"pressure_hpa":938.849,"pm1_0":1.58,"pm2_5":1.64,"nc0_5":10.87,"nc1_0":1.694,"nc2_5":0.038,"cnt":24905,"fire_alarm":true,"recorded_at":"2022-06-09T07:03:56+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":24970,"utc_seconds":1654758301,"temperature_c":18.164,"humidity_pct":49.49,"tvoc_ppb":1272,"eco2_ppm":401,"raw_h2":12993,"raw_ethanol":19411,"pressure_hpa":938.866,"pm1_0":1.78,"pm2_5":1.85,"nc0_5":12.28,"nc1_0":1.915,"nc2_5":0.043,"cnt":24970,"fire_alarm":true,"recorded_at":"2022-06-09T07:05:01+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":25398,"utc_seconds":1654761747,"temperature_c":13.56,"humidity_pct":53.34,"tvoc_ppb":8,"eco2_ppm":400,"raw_h2":12726,"raw_ethanol":19762,"pressure_hpa":939.837,"pm1_0":0.21,"pm2_5":0.27,"nc0_5":1.33,"nc1_0":0.287,"nc2_5":0.054,"cnt":404,"fire_alarm":false,"recorded_at":"2022-06-09T08:02:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":25722,"utc_seconds":1654762071,"temperature_c":12.42,"humidity_pct":55.66,"tvoc_ppb":4,"eco2_ppm":400,"raw_h2":12921,"raw_ethanol":19858,"pressure_hpa":939.745,"pm1_0":0.31,"pm2_5":0.32,"nc0_5":2.15,"nc1_0":0.336,"nc2_5":0.008,"cnt":728,"fire_alarm":false,"recorded_at":"2022-06-09T08:07:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":26620,"utc_seconds":1654762969,"temperature_c":16.28,"humidity_pct":52.07,"tvoc_ppb":21,"eco2_ppm":400,"raw_h2":13117,"raw_ethanol":20013,"pressure_hpa":939.603,"pm1_0":0.69,"pm2_5":0.71,"nc0_5":4.72,"nc1_0":0.736,"nc2_5":0.017,"cnt":1626,"fire_alarm":false,"recorded_at":"2022-06-09T08:22:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":26718,"utc_seconds":1654763067,"temperature_c":18.55,"humidity_pct":49.73,"tvoc_ppb":9,"eco2_ppm":400,"raw_h2":13144,"raw_ethanol":20053,"pressure_hpa":939.614,"pm1_0":0.55,"pm2_5":0.57,"nc0_5":3.76,"nc1_0":0.586,"nc2_5":0.013,"cnt":1724,"fire_alarm":false,"recorded_at":"2022-06-09T08:24:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":26981,"utc_seconds":1654763330,"temperature_c":16.99,"humidity_pct":52.48,"tvoc_ppb":73,"eco2_ppm":400,"raw_h2":13106,"raw_ethanol":19974,"pressure_hpa":939.604,"pm1_0":0.71,"pm2_5":0.74,"nc0_5":4.91,"nc1_0":0.765,"nc2_5":0.017,"cnt":1987,"fire_alarm":false,"recorded_at":"2022-06-09T08:28:50+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":27509,"utc_seconds":1654763858,"temperature_c":18.16,"humidity_pct":51.95,"tvoc_ppb":62,"eco2_ppm":400,"raw_h2":13191,"raw_ethanol":20115,"pressure_hpa":939.616,"pm1_0":0.97,"pm2_5":1.01,"nc0_5":6.68,"nc1_0":1.041,"nc2_5":0.024,"cnt":2515,"fire_alarm":false,"recorded_at":"2022-06-09T08:37:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":27726,"utc_seconds":1654764075,"temperature_c":20.84,"humidity_pct":46.29,"tvoc_ppb":101,"eco2_ppm":400,"raw_h2":13188,"raw_ethanol":20090,"pressure_hpa":939.638,"pm1_0":0.98,"pm2_5":1.02,"nc0_5":6.75,"nc1_0":1.053,"nc2_5":0.024,"cnt":2732,"fire_alarm":false,"recorded_at":"2022-06-09T08:41:15+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":27977,"utc_seconds":1654764326,"temperature_c":18.86,"humidity_pct":49.61,"tvoc_ppb":183,"eco2_ppm":400,"raw_h2":13138,"raw_ethanol":19988,"pressure_hpa":939.651,"pm1_0":0.92,"pm2_5":0.96,"nc0_5":6.33,"nc1_0":0.987,"nc2_5":0.022,"cnt":2983,"fire_alarm":false,"recorded_at":"2022-06-09T08:45:26+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28052,"utc_seconds":1654764401,"temperature_c":18.58,"humidity_pct":51.79,"tvoc_ppb":165,"eco2_ppm":400,"raw_h2":13162,"raw_ethanol":20008,"pressure_hpa":939.675,"pm1_0":0.87,"pm2_5":0.9,"nc0_5":5.96,"nc1_0":0.93,"nc2_5":0.021,"cnt":3058,"fire_alarm":false,"recorded_at":"2022-06-09T08:46:41+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28326,"utc_seconds":1654764675,"temperature_c":22.22,"humidity_pct":56.04,"tvoc_ppb":237,"eco2_ppm":400,"raw_h2":13167,"raw_ethanol":19984,"pressure_hpa":939.654,"pm1_0":2.41,"pm2_5":2.5,"nc0_5":16.58,"nc1_0":2.585,"nc2_5":0.058,"cnt":3332,"fire_alarm":true,"recorded_at":"2022-06-09T08:51:15+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28814,"utc_seconds":1654765163,"temperature_c":20.56,"humidity_pct":49.67,"tvoc_ppb":165,"eco2_ppm":400,"raw_h2":13203,"raw_ethanol":20045,"pressure_hpa":939.682,"pm1_0":1.98,"pm2_5":2.06,"nc0_5":13.65,"nc1_0":2.128,"nc2_5":0.048,"cnt":3820,"fire_alarm":true,"recorded_at":"2022-06-09T08:59:23+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28854,"utc_seconds":1654765203,"temperature_c":22.4,"humidity_pct":47.03,"tvoc_ppb":163,"eco2_ppm":400,"raw_h2":13214,"raw_ethanol":20050,"pressure_hpa":939.668,"pm1_0":2.07,"pm2_5":2.16,"nc0_5":14.28,"nc1_0":2.227,"nc2_5":0.05,"cnt":3860,"fire_alarm":true,"recorded_at":"2022-06-09T09:00:03+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28874,"utc_seconds":1654765223,"temperature_c":20.54,"humidity_pct":49.85,"tvoc_ppb":182,"eco2_ppm":400,"raw_h2":13203,"raw_ethanol":20033,"pressure_hpa":939.672,"pm1_0":2.25,"pm2_5":2.33,"nc0_5":15.46,"nc1_0":2.411,"nc2_5":0.054,"cnt":3880,"fire_alarm":true,"recorded_at":"2022-06-09T09:00:23+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28896,"utc_seconds":1654765245,"temperature_c":19.95,"humidity_pct":52.94,"tvoc_ppb":177,"eco2_ppm":400,"raw_h2":13202,"raw_ethanol":20040,"pressure_hpa":939.68,"pm1_0":2.2,"pm2_5":2.29,"nc0_5":15.14,"nc1_0":2.361,"nc2_5":0.053,"cnt":3902,"fire_alarm":true,"recorded_at":"2022-06-09T09:00:45+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":28915,"utc_seconds":1654765264,"temperature_c":22.42,"humidity_pct":47.25,"tvoc_ppb":158,"eco2_ppm":400,"raw_h2":13210,"raw_ethanol":20052,"pressure_hpa":939.66,"pm1_0":2.22,"pm2_5":2.3,"nc0_5":15.25,"nc1_0":2.378,"nc2_5":0.054,"cnt":3921,"fire_alarm":true,"recorded_at":"2022-06-09T09:01:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29063,"utc_seconds":1654765412,"temperature_c":20.06,"humidity_pct":52.82,"tvoc_ppb":123,"eco2_ppm":400,"raw_h2":13224,"raw_ethanol":20093,"pressure_hpa":939.622,"pm1_0":2.14,"pm2_5":2.23,"nc0_5":14.74,"nc1_0":2.299,"nc2_5":0.052,"cnt":4069,"fire_alarm":true,"recorded_at":"2022-06-09T09:03:32+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29070,"utc_seconds":1654765419,"temperature_c":19.95,"humidity_pct":53.41,"tvoc_ppb":123,"eco2_ppm":400,"raw_h2":13229,"raw_ethanol":20100,"pressure_hpa":939.619,"pm1_0":2.28,"pm2_5":2.37,"nc0_5":15.71,"nc1_0":2.45,"nc2_5":0.055,"cnt":4076,"fire_alarm":true,"recorded_at":"2022-06-09T09:03:39+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29179,"utc_seconds":1654765528,"temperature_c":20.16,"humidity_pct":52.37,"tvoc_ppb":86,"eco2_ppm":400,"raw_h2":13240,"raw_ethanol":20124,"pressure_hpa":939.639,"pm1_0":2.35,"pm2_5":2.44,"nc0_5":16.14,"nc1_0":2.517,"nc2_5":0.057,"cnt":4185,"fire_alarm":true,"recorded_at":"2022-06-09T09:05:28+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29274,"utc_seconds":1654765623,"temperature_c":22.33,"humidity_pct":47.97,"tvoc_ppb":96,"eco2_ppm":400,"raw_h2":13227,"raw_ethanol":20114,"pressure_hpa":939.66,"pm1_0":1.8,"pm2_5":1.87,"nc0_5":12.36,"nc1_0":1.927,"nc2_5":0.044,"cnt":4280,"fire_alarm":true,"recorded_at":"2022-06-09T09:07:03+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29309,"utc_seconds":1654765658,"temperature_c":19.92,"humidity_pct":53.58,"tvoc_ppb":108,"eco2_ppm":400,"raw_h2":13227,"raw_ethanol":20103,"pressure_hpa":939.647,"pm1_0":2.1,"pm2_5":2.18,"nc0_5":14.45,"nc1_0":2.253,"nc2_5":0.051,"cnt":4315,"fire_alarm":true,"recorded_at":"2022-06-09T09:07:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29405,"utc_seconds":1654765754,"temperature_c":22.49,"humidity_pct":47.65,"tvoc_ppb":88,"eco2_ppm":400,"raw_h2":13244,"raw_ethanol":20129,"pressure_hpa":939.659,"pm1_0":1.98,"pm2_5":2.06,"nc0_5":13.64,"nc1_0":2.128,"nc2_5":0.048,"cnt":4411,"fire_alarm":true,"recorded_at":"2022-06-09T09:09:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":29976,"utc_seconds":1654766325,"temperature_c":20.92,"humidity_pct":54.93,"tvoc_ppb":28,"eco2_ppm":400,"raw_h2":13254,"raw_ethanol":20191,"pressure_hpa":939.701,"pm1_0":2.06,"pm2_5":2.14,"nc0_5":14.16,"nc1_0":2.208,"nc2_5":0.05,"cnt":4982,"fire_alarm":true,"recorded_at":"2022-06-09T09:18:45+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":30295,"utc_seconds":1654766644,"temperature_c":22.17,"humidity_pct":48.67,"tvoc_ppb":20,"eco2_ppm":400,"raw_h2":13254,"raw_ethanol":20210,"pressure_hpa":939.684,"pm1_0":2.28,"pm2_5":2.37,"nc0_5":15.68,"nc1_0":2.445,"nc2_5":0.055,"cnt":5301,"fire_alarm":true,"recorded_at":"2022-06-09T09:24:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":30725,"utc_seconds":1654767074,"temperature_c":20.61,"humidity_pct":49.23,"tvoc_ppb":62,"eco2_ppm":400,"raw_h2":13234,"raw_ethanol":20191,"pressure_hpa":939.687,"pm1_0":2.47,"pm2_5":2.56,"nc0_5":16.98,"nc1_0":2.649,"nc2_5":0.06,"cnt":5731,"fire_alarm":true,"recorded_at":"2022-06-09T09:31:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":31394,"utc_seconds":1654767743,"temperature_c":19.33,"humidity_pct":54.03,"tvoc_ppb":189,"eco2_ppm":400,"raw_h2":13156,"raw_ethanol":20065,"pressure_hpa":939.638,"pm1_0":0.86,"pm2_5":0.89,"nc0_5":5.91,"nc1_0":0.922,"nc2_5":0.021,"cnt":6400,"fire_alarm":true,"recorded_at":"2022-06-09T09:42:23+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":31409,"utc_seconds":1654767758,"temperature_c":18.96,"humidity_pct":55.63,"tvoc_ppb":194,"eco2_ppm":400,"raw_h2":13153,"raw_ethanol":20073,"pressure_hpa":939.641,"pm1_0":0.78,"pm2_5":0.81,"nc0_5":5.36,"nc1_0":0.835,"nc2_5":0.019,"cnt":6415,"fire_alarm":true,"recorded_at":"2022-06-09T09:42:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":31649,"utc_seconds":1654767998,"temperature_c":18.75,"humidity_pct":55.9,"tvoc_ppb":204,"eco2_ppm":400,"raw_h2":13147,"raw_ethanol":20057,"pressure_hpa":939.581,"pm1_0":0.6,"pm2_5":0.63,"nc0_5":4.16,"nc1_0":0.649,"nc2_5":0.015,"cnt":6655,"fire_alarm":true,"recorded_at":"2022-06-09T09:46:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":31792,"utc_seconds":1654768141,"temperature_c":21.18,"humidity_pct":49.92,"tvoc_ppb":200,"eco2_ppm":400,"raw_h2":13155,"raw_ethanol":20061,"pressure_hpa":939.625,"pm1_0":0.53,"pm2_5":0.55,"nc0_5":3.61,"nc1_0":0.564,"nc2_5":0.013,"cnt":6798,"fire_alarm":true,"recorded_at":"2022-06-09T09:49:01+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":31801,"utc_seconds":1654768150,"temperature_c":21.35,"humidity_pct":49.42,"tvoc_ppb":213,"eco2_ppm":400,"raw_h2":13140,"raw_ethanol":20041,"pressure_hpa":939.619,"pm1_0":0.71,"pm2_5":0.73,"nc0_5":4.86,"nc1_0":0.757,"nc2_5":0.017,"cnt":6807,"fire_alarm":true,"recorded_at":"2022-06-09T09:49:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":32730,"utc_seconds":1654769079,"temperature_c":18.86,"humidity_pct":53.27,"tvoc_ppb":299,"eco2_ppm":400,"raw_h2":13110,"raw_ethanol":19971,"pressure_hpa":939.397,"pm1_0":0.38,"pm2_5":0.4,"nc0_5":2.63,"nc1_0":0.411,"nc2_5":0.009,"cnt":7736,"fire_alarm":true,"recorded_at":"2022-06-09T10:04:39+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":32877,"utc_seconds":1654769226,"temperature_c":20.71,"humidity_pct":50.85,"tvoc_ppb":302,"eco2_ppm":400,"raw_h2":13111,"raw_ethanol":19964,"pressure_hpa":939.419,"pm1_0":0.42,"pm2_5":0.44,"nc0_5":2.92,"nc1_0":0.456,"nc2_5":0.01,"cnt":7883,"fire_alarm":true,"recorded_at":"2022-06-09T10:07:06+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":32896,"utc_seconds":1654769245,"temperature_c":18.79,"humidity_pct":53.6,"tvoc_ppb":310,"eco2_ppm":400,"raw_h2":13106,"raw_ethanol":19958,"pressure_hpa":939.407,"pm1_0":0.31,"pm2_5":0.32,"nc0_5":2.12,"nc1_0":0.33,"nc2_5":0.007,"cnt":7902,"fire_alarm":true,"recorded_at":"2022-06-09T10:07:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":33002,"utc_seconds":1654769351,"temperature_c":19.78,"humidity_pct":49.73,"tvoc_ppb":327,"eco2_ppm":400,"raw_h2":13094,"raw_ethanol":19940,"pressure_hpa":939.394,"pm1_0":0.29,"pm2_5":0.3,"nc0_5":2.02,"nc1_0":0.314,"nc2_5":0.007,"cnt":8008,"fire_alarm":true,"recorded_at":"2022-06-09T10:09:11+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":33035,"utc_seconds":1654769384,"temperature_c":20.48,"humidity_pct":52.33,"tvoc_ppb":303,"eco2_ppm":400,"raw_h2":13114,"raw_ethanol":19961,"pressure_hpa":939.385,"pm1_0":0.28,"pm2_5":0.29,"nc0_5":1.95,"nc1_0":0.305,"nc2_5":0.007,"cnt":8041,"fire_alarm":true,"recorded_at":"2022-06-09T10:09:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":33340,"utc_seconds":1654769689,"temperature_c":19.43,"humidity_pct":50.48,"tvoc_ppb":338,"eco2_ppm":400,"raw_h2":13088,"raw_ethanol":19927,"pressure_hpa":939.393,"pm1_0":0.26,"pm2_5":0.27,"nc0_5":1.79,"nc1_0":0.279,"nc2_5":0.006,"cnt":8346,"fire_alarm":true,"recorded_at":"2022-06-09T10:14:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":33806,"utc_seconds":1654770155,"temperature_c":18.36,"humidity_pct":57.26,"tvoc_ppb":362,"eco2_ppm":400,"raw_h2":13064,"raw_ethanol":19894,"pressure_hpa":939.272,"pm1_0":0.44,"pm2_5":0.46,"nc0_5":3.04,"nc1_0":0.475,"nc2_5":0.011,"cnt":8812,"fire_alarm":true,"recorded_at":"2022-06-09T10:22:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":33957,"utc_seconds":1654770306,"temperature_c":19.23,"humidity_pct":51.92,"tvoc_ppb":370,"eco2_ppm":400,"raw_h2":13050,"raw_ethanol":19883,"pressure_hpa":939.271,"pm1_0":0.43,"pm2_5":0.44,"nc0_5":2.94,"nc1_0":0.458,"nc2_5":0.01,"cnt":8963,"fire_alarm":true,"recorded_at":"2022-06-09T10:25:06+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34350,"utc_seconds":1654770699,"temperature_c":19.95,"humidity_pct":62.87,"tvoc_ppb":621,"eco2_ppm":554,"raw_h2":12863,"raw_ethanol":19637,"pressure_hpa":939.246,"pm1_0":2.5,"pm2_5":2.6,"nc0_5":17.21,"nc1_0":2.683,"nc2_5":0.061,"cnt":9356,"fire_alarm":true,"recorded_at":"2022-06-09T10:31:39+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34388,"utc_seconds":1654770737,"temperature_c":22.11,"humidity_pct":58.63,"tvoc_ppb":626,"eco2_ppm":547,"raw_h2":12865,"raw_ethanol":19629,"pressure_hpa":939.189,"pm1_0":2.45,"pm2_5":2.55,"nc0_5":16.88,"nc1_0":2.632,"nc2_5":0.059,"cnt":9394,"fire_alarm":true,"recorded_at":"2022-06-09T10:32:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34392,"utc_seconds":1654770741,"temperature_c":22.25,"humidity_pct":58.09,"tvoc_ppb":621,"eco2_ppm":544,"raw_h2":12874,"raw_ethanol":19632,"pressure_hpa":939.193,"pm1_0":2.67,"pm2_5":2.78,"nc0_5":18.39,"nc1_0":2.867,"nc2_5":0.065,"cnt":9398,"fire_alarm":true,"recorded_at":"2022-06-09T10:32:21+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34396,"utc_seconds":1654770745,"temperature_c":22.33,"humidity_pct":57.73,"tvoc_ppb":623,"eco2_ppm":544,"raw_h2":12877,"raw_ethanol":19638,"pressure_hpa":939.215,"pm1_0":2.97,"pm2_5":3.09,"nc0_5":20.46,"nc1_0":3.191,"nc2_5":0.072,"cnt":9402,"fire_alarm":true,"recorded_at":"2022-06-09T10:32:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34508,"utc_seconds":1654770857,"temperature_c":22.86,"humidity_pct":57.21,"tvoc_ppb":676,"eco2_ppm":682,"raw_h2":12792,"raw_ethanol":19581,"pressure_hpa":939.175,"pm1_0":2.38,"pm2_5":2.47,"nc0_5":16.37,"nc1_0":2.553,"nc2_5":0.058,"cnt":9514,"fire_alarm":true,"recorded_at":"2022-06-09T10:34:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34615,"utc_seconds":1654770964,"temperature_c":22.8,"humidity_pct":60.24,"tvoc_ppb":687,"eco2_ppm":701,"raw_h2":12782,"raw_ethanol":19564,"pressure_hpa":939.132,"pm1_0":2.89,"pm2_5":3.0,"nc0_5":19.9,"nc1_0":3.104,"nc2_5":0.07,"cnt":9621,"fire_alarm":true,"recorded_at":"2022-06-09T10:36:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":34853,"utc_seconds":1654771202,"temperature_c":23.41,"humidity_pct":57.22,"tvoc_ppb":689,"eco2_ppm":822,"raw_h2":12716,"raw_ethanol":19567,"pressure_hpa":939.133,"pm1_0":2.46,"pm2_5":2.55,"nc0_5":16.9,"nc1_0":2.635,"nc2_5":0.06,"cnt":9859,"fire_alarm":true,"recorded_at":"2022-06-09T10:40:02+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":35386,"utc_seconds":1654771735,"temperature_c":23.17,"humidity_pct":55.81,"tvoc_ppb":906,"eco2_ppm":685,"raw_h2":12785,"raw_ethanol":19505,"pressure_hpa":939.097,"pm1_0":2.52,"pm2_5":2.62,"nc0_5":17.36,"nc1_0":2.707,"nc2_5":0.061,"cnt":10392,"fire_alarm":true,"recorded_at":"2022-06-09T10:48:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":35557,"utc_seconds":1654771906,"temperature_c":23.63,"humidity_pct":54.58,"tvoc_ppb":954,"eco2_ppm":686,"raw_h2":12784,"raw_ethanol":19492,"pressure_hpa":939.096,"pm1_0":2.25,"pm2_5":2.34,"nc0_5":15.48,"nc1_0":2.415,"nc2_5":0.055,"cnt":10563,"fire_alarm":true,"recorded_at":"2022-06-09T10:51:46+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":36027,"utc_seconds":1654772376,"temperature_c":24.47,"humidity_pct":51.65,"tvoc_ppb":1055,"eco2_ppm":664,"raw_h2":12788,"raw_ethanol":19467,"pressure_hpa":939.097,"pm1_0":2.33,"pm2_5":2.42,"nc0_5":16.06,"nc1_0":2.504,"nc2_5":0.057,"cnt":11033,"fire_alarm":true,"recorded_at":"2022-06-09T10:59:36+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":36526,"utc_seconds":1654772875,"temperature_c":26.36,"humidity_pct":48.93,"tvoc_ppb":1025,"eco2_ppm":605,"raw_h2":12819,"raw_ethanol":19471,"pressure_hpa":939.057,"pm1_0":1.85,"pm2_5":1.92,"nc0_5":12.71,"nc1_0":1.983,"nc2_5":0.045,"cnt":11532,"fire_alarm":true,"recorded_at":"2022-06-09T11:07:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":36752,"utc_seconds":1654773101,"temperature_c":24.22,"humidity_pct":53.55,"tvoc_ppb":1051,"eco2_ppm":610,"raw_h2":12812,"raw_ethanol":19464,"pressure_hpa":939.082,"pm1_0":2.02,"pm2_5":2.09,"nc0_5":13.87,"nc1_0":2.164,"nc2_5":0.049,"cnt":11758,"fire_alarm":true,"recorded_at":"2022-06-09T11:11:41+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":36771,"utc_seconds":1654773120,"temperature_c":26.69,"humidity_pct":47.75,"tvoc_ppb":1051,"eco2_ppm":599,"raw_h2":12812,"raw_ethanol":19465,"pressure_hpa":939.084,"pm1_0":1.95,"pm2_5":2.03,"nc0_5":13.42,"nc1_0":2.092,"nc2_5":0.047,"cnt":11777,"fire_alarm":true,"recorded_at":"2022-06-09T11:12:00+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":37457,"utc_seconds":1654773806,"temperature_c":24.72,"humidity_pct":52.15,"tvoc_ppb":1099,"eco2_ppm":566,"raw_h2":12823,"raw_ethanol":19452,"pressure_hpa":939.015,"pm1_0":2.2,"pm2_5":2.29,"nc0_5":15.17,"nc1_0":2.365,"nc2_5":0.053,"cnt":12463,"fire_alarm":true,"recorded_at":"2022-06-09T11:23:26+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":37665,"utc_seconds":1654774014,"temperature_c":26.71,"humidity_pct":48.18,"tvoc_ppb":1088,"eco2_ppm":556,"raw_h2":12831,"raw_ethanol":19467,"pressure_hpa":938.978,"pm1_0":1.96,"pm2_5":2.04,"nc0_5":13.52,"nc1_0":2.109,"nc2_5":0.048,"cnt":12671,"fire_alarm":true,"recorded_at":"2022-06-09T11:26:54+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":37711,"utc_seconds":1654774060,"temperature_c":24.44,"humidity_pct":53.2,"tvoc_ppb":1096,"eco2_ppm":569,"raw_h2":12831,"raw_ethanol":19456,"pressure_hpa":938.965,"pm1_0":1.76,"pm2_5":1.83,"nc0_5":12.09,"nc1_0":1.885,"nc2_5":0.043,"cnt":12717,"fire_alarm":true,"recorded_at":"2022-06-09T11:27:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":37733,"utc_seconds":1654774082,"temperature_c":26.91,"humidity_pct":47.38,"tvoc_ppb":1082,"eco2_ppm":557,"raw_h2":12830,"raw_ethanol":19457,"pressure_hpa":938.963,"pm1_0":2.03,"pm2_5":2.11,"nc0_5":13.98,"nc1_0":2.179,"nc2_5":0.049,"cnt":12739,"fire_alarm":true,"recorded_at":"2022-06-09T11:28:02+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":37797,"utc_seconds":1654774146,"temperature_c":27.01,"humidity_pct":47.18,"tvoc_ppb":1011,"eco2_ppm":538,"raw_h2":12836,"raw_ethanol":19472,"pressure_hpa":938.938,"pm1_0":2.14,"pm2_5":2.22,"nc0_5":14.7,"nc1_0":2.292,"nc2_5":0.052,"cnt":12803,"fire_alarm":true,"recorded_at":"2022-06-09T11:29:06+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":38287,"utc_seconds":1654774636,"temperature_c":25.41,"humidity_pct":48.37,"tvoc_ppb":1159,"eco2_ppm":555,"raw_h2":12825,"raw_ethanol":19431,"pressure_hpa":938.918,"pm1_0":1.71,"pm2_5":1.77,"nc0_5":11.75,"nc1_0":1.833,"nc2_5":0.041,"cnt":13293,"fire_alarm":true,"recorded_at":"2022-06-09T11:37:16+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":38502,"utc_seconds":1654774851,"temperature_c":26.64,"humidity_pct":48.91,"tvoc_ppb":1086,"eco2_ppm":502,"raw_h2":12848,"raw_ethanol":19456,"pressure_hpa":938.873,"pm1_0":1.94,"pm2_5":2.01,"nc0_5":13.35,"nc1_0":2.081,"nc2_5":0.047,"cnt":13508,"fire_alarm":true,"recorded_at":"2022-06-09T11:40:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":38703,"utc_seconds":1654775052,"temperature_c":27.15,"humidity_pct":46.91,"tvoc_ppb":1128,"eco2_ppm":495,"raw_h2":12847,"raw_ethanol":19456,"pressure_hpa":938.884,"pm1_0":2.13,"pm2_5":2.21,"nc0_5":14.64,"nc1_0":2.282,"nc2_5":0.052,"cnt":13709,"fire_alarm":true,"recorded_at":"2022-06-09T11:44:12+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":38705,"utc_seconds":1654775054,"temperature_c":25.99,"humidity_pct":46.89,"tvoc_ppb":1216,"eco2_ppm":529,"raw_h2":12832,"raw_ethanol":19428,"pressure_hpa":938.879,"pm1_0":2.13,"pm2_5":2.21,"nc0_5":14.67,"nc1_0":2.287,"nc2_5":0.052,"cnt":13711,"fire_alarm":true,"recorded_at":"2022-06-09T11:44:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39130,"utc_seconds":1654775479,"temperature_c":25.1,"humidity_pct":50.46,"tvoc_ppb":1155,"eco2_ppm":500,"raw_h2":12853,"raw_ethanol":19447,"pressure_hpa":938.907,"pm1_0":1.86,"pm2_5":1.93,"nc0_5":12.78,"nc1_0":1.993,"nc2_5":0.045,"cnt":14136,"fire_alarm":true,"recorded_at":"2022-06-09T11:51:19+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39155,"utc_seconds":1654775504,"temperature_c":25.7,"humidity_pct":53.46,"tvoc_ppb":1088,"eco2_ppm":468,"raw_h2":12869,"raw_ethanol":19475,"pressure_hpa":938.9,"pm1_0":1.72,"pm2_5":1.79,"nc0_5":11.86,"nc1_0":1.85,"nc2_5":0.042,"cnt":14161,"fire_alarm":true,"recorded_at":"2022-06-09T11:51:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39360,"utc_seconds":1654775709,"temperature_c":27.11,"humidity_pct":47.07,"tvoc_ppb":1140,"eco2_ppm":470,"raw_h2":12860,"raw_ethanol":19444,"pressure_hpa":938.904,"pm1_0":1.84,"pm2_5":1.91,"nc0_5":12.68,"nc1_0":1.977,"nc2_5":0.045,"cnt":14366,"fire_alarm":true,"recorded_at":"2022-06-09T11:55:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39415,"utc_seconds":1654775764,"temperature_c":27.01,"humidity_pct":47.29,"tvoc_ppb":1141,"eco2_ppm":477,"raw_h2":12860,"raw_ethanol":19439,"pressure_hpa":938.898,"pm1_0":1.63,"pm2_5":1.7,"nc0_5":11.23,"nc1_0":1.751,"nc2_5":0.04,"cnt":14421,"fire_alarm":true,"recorded_at":"2022-06-09T11:56:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39442,"utc_seconds":1654775791,"temperature_c":24.66,"humidity_pct":52.83,"tvoc_ppb":1159,"eco2_ppm":480,"raw_h2":12854,"raw_ethanol":19438,"pressure_hpa":938.911,"pm1_0":1.95,"pm2_5":2.03,"nc0_5":13.42,"nc1_0":2.092,"nc2_5":0.047,"cnt":14448,"fire_alarm":true,"recorded_at":"2022-06-09T11:56:31+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39699,"utc_seconds":1654776048,"temperature_c":26.43,"humidity_pct":50.38,"tvoc_ppb":1082,"eco2_ppm":449,"raw_h2":12864,"raw_ethanol":19453,"pressure_hpa":938.887,"pm1_0":1.48,"pm2_5":1.54,"nc0_5":10.21,"nc1_0":1.593,"nc2_5":0.036,"cnt":14705,"fire_alarm":true,"recorded_at":"2022-06-09T12:00:48+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39730,"utc_seconds":1654776079,"temperature_c":25.12,"humidity_pct":50.54,"tvoc_ppb":1117,"eco2_ppm":454,"raw_h2":12862,"raw_ethanol":19440,"pressure_hpa":938.826,"pm1_0":1.75,"pm2_5":1.82,"nc0_5":12.06,"nc1_0":1.88,"nc2_5":0.042,"cnt":14736,"fire_alarm":true,"recorded_at":"2022-06-09T12:01:19+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39825,"utc_seconds":1654776174,"temperature_c":26.74,"humidity_pct":48.32,"tvoc_ppb":1140,"eco2_ppm":450,"raw_h2":12870,"raw_ethanol":19450,"pressure_hpa":938.841,"pm1_0":1.91,"pm2_5":1.98,"nc0_5":13.14,"nc1_0":2.05,"nc2_5":0.046,"cnt":14831,"fire_alarm":true,"recorded_at":"2022-06-09T12:02:54+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":39882,"utc_seconds":1654776231,"temperature_c":26.6,"humidity_pct":49.02,"tvoc_ppb":1107,"eco2_ppm":445,"raw_h2":12868,"raw_ethanol":19451,"pressure_hpa":938.811,"pm1_0":2.04,"pm2_5":2.12,"nc0_5":14.03,"nc1_0":2.188,"nc2_5":0.049,"cnt":14888,"fire_alarm":true,"recorded_at":"2022-06-09T12:03:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":40055,"utc_seconds":1654776404,"temperature_c":25.67,"humidity_pct":53.63,"tvoc_ppb":996,"eco2_ppm":415,"raw_h2":12881,"raw_ethanol":19472,"pressure_hpa":938.81,"pm1_0":1.95,"pm2_5":2.03,"nc0_5":13.45,"nc1_0":2.097,"nc2_5":0.047,"cnt":15061,"fire_alarm":true,"recorded_at":"2022-06-09T12:06:44+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":40373,"utc_seconds":1654776722,"temperature_c":26.96,"humidity_pct":47.53,"tvoc_ppb":1151,"eco2_ppm":422,"raw_h2":12874,"raw_ethanol":19442,"pressure_hpa":938.826,"pm1_0":1.72,"pm2_5":1.79,"nc0_5":11.85,"nc1_0":1.848,"nc2_5":0.042,"cnt":15379,"fire_alarm":true,"recorded_at":"2022-06-09T12:12:02+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":40936,"utc_seconds":1654777285,"temperature_c":24.79,"humidity_pct":52.43,"tvoc_ppb":1159,"eco2_ppm":415,"raw_h2":12884,"raw_ethanol":19442,"pressure_hpa":938.768,"pm1_0":1.91,"pm2_5":1.98,"nc0_5":13.14,"nc1_0":2.049,"nc2_5":0.046,"cnt":15942,"fire_alarm":true,"recorded_at":"2022-06-09T12:21:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":41026,"utc_seconds":1654777375,"temperature_c":26.78,"humidity_pct":48.3,"tvoc_ppb":1130,"eco2_ppm":401,"raw_h2":12886,"raw_ethanol":19450,"pressure_hpa":938.779,"pm1_0":1.86,"pm2_5":1.93,"nc0_5":12.81,"nc1_0":1.997,"nc2_5":0.045,"cnt":16032,"fire_alarm":true,"recorded_at":"2022-06-09T12:22:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":41610,"utc_seconds":1654777959,"temperature_c":24.51,"humidity_pct":53.57,"tvoc_ppb":1186,"eco2_ppm":416,"raw_h2":12882,"raw_ethanol":19429,"pressure_hpa":938.776,"pm1_0":1.75,"pm2_5":1.82,"nc0_5":12.07,"nc1_0":1.881,"nc2_5":0.042,"cnt":16616,"fire_alarm":true,"recorded_at":"2022-06-09T12:32:39+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":41683,"utc_seconds":1654778032,"temperature_c":26.64,"humidity_pct":48.98,"tvoc_ppb":1055,"eco2_ppm":400,"raw_h2":12899,"raw_ethanol":19463,"pressure_hpa":938.778,"pm1_0":2.08,"pm2_5":2.17,"nc0_5":14.34,"nc1_0":2.237,"nc2_5":0.051,"cnt":16689,"fire_alarm":true,"recorded_at":"2022-06-09T12:33:52+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":41849,"utc_seconds":1654778198,"temperature_c":24.48,"humidity_pct":53.61,"tvoc_ppb":1122,"eco2_ppm":406,"raw_h2":12888,"raw_ethanol":19445,"pressure_hpa":938.749,"pm1_0":1.71,"pm2_5":1.77,"nc0_5":11.75,"nc1_0":1.832,"nc2_5":0.041,"cnt":16855,"fire_alarm":true,"recorded_at":"2022-06-09T12:36:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42044,"utc_seconds":1654778393,"temperature_c":26.69,"humidity_pct":48.78,"tvoc_ppb":1140,"eco2_ppm":400,"raw_h2":12895,"raw_ethanol":19438,"pressure_hpa":938.789,"pm1_0":2.06,"pm2_5":2.14,"nc0_5":14.17,"nc1_0":2.209,"nc2_5":0.05,"cnt":17050,"fire_alarm":true,"recorded_at":"2022-06-09T12:39:53+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42246,"utc_seconds":1654778595,"temperature_c":25.55,"humidity_pct":47.94,"tvoc_ppb":1212,"eco2_ppm":420,"raw_h2":12887,"raw_ethanol":19425,"pressure_hpa":938.792,"pm1_0":1.59,"pm2_5":1.65,"nc0_5":10.95,"nc1_0":1.708,"nc2_5":0.039,"cnt":17252,"fire_alarm":true,"recorded_at":"2022-06-09T12:43:15+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42504,"utc_seconds":1654778853,"temperature_c":24.59,"humidity_pct":53.36,"tvoc_ppb":1130,"eco2_ppm":401,"raw_h2":12895,"raw_ethanol":19445,"pressure_hpa":938.736,"pm1_0":1.68,"pm2_5":1.75,"nc0_5":11.56,"nc1_0":1.803,"nc2_5":0.041,"cnt":17510,"fire_alarm":true,"recorded_at":"2022-06-09T12:47:33+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42660,"utc_seconds":1654779009,"temperature_c":27.11,"humidity_pct":47.43,"tvoc_ppb":1117,"eco2_ppm":400,"raw_h2":12908,"raw_ethanol":19447,"pressure_hpa":938.727,"pm1_0":1.72,"pm2_5":1.78,"nc0_5":11.81,"nc1_0":1.842,"nc2_5":0.042,"cnt":17666,"fire_alarm":true,"recorded_at":"2022-06-09T12:50:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42710,"utc_seconds":1654779059,"temperature_c":26.88,"humidity_pct":48.06,"tvoc_ppb":1138,"eco2_ppm":409,"raw_h2":12906,"raw_ethanol":19444,"pressure_hpa":938.706,"pm1_0":1.69,"pm2_5":1.76,"nc0_5":11.65,"nc1_0":1.817,"nc2_5":0.041,"cnt":17716,"fire_alarm":true,"recorded_at":"2022-06-09T12:50:59+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42726,"utc_seconds":1654779075,"temperature_c":25.54,"humidity_pct":48.06,"tvoc_ppb":1216,"eco2_ppm":414,"raw_h2":12895,"raw_ethanol":19424,"pressure_hpa":938.712,"pm1_0":1.4,"pm2_5":1.45,"nc0_5":9.62,"nc1_0":1.5,"nc2_5":0.034,"cnt":17732,"fire_alarm":true,"recorded_at":"2022-06-09T12:51:15+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":42785,"utc_seconds":1654779134,"temperature_c":25.95,"humidity_pct":47.27,"tvoc_ppb":1117,"eco2_ppm":410,"raw_h2":12895,"raw_ethanol":19444,"pressure_hpa":938.721,"pm1_0":1.87,"pm2_5":1.94,"nc0_5":12.84,"nc1_0":2.002,"nc2_5":0.045,"cnt":17791,"fire_alarm":true,"recorded_at":"2022-06-09T12:52:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":43283,"utc_seconds":1654779632,"temperature_c":24.61,"humidity_pct":53.27,"tvoc_ppb":1182,"eco2_ppm":416,"raw_h2":12907,"raw_ethanol":19436,"pressure_hpa":938.684,"pm1_0":1.7,"pm2_5":1.77,"nc0_5":11.71,"nc1_0":1.826,"nc2_5":0.041,"cnt":18289,"fire_alarm":true,"recorded_at":"2022-06-09T13:00:32+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":43346,"utc_seconds":1654779695,"temperature_c":24.57,"humidity_pct":53.49,"tvoc_ppb":1187,"eco2_ppm":407,"raw_h2":12909,"raw_ethanol":19430,"pressure_hpa":938.755,"pm1_0":1.45,"pm2_5":1.51,"nc0_5":9.97,"nc1_0":1.555,"nc2_5":0.035,"cnt":18352,"fire_alarm":true,"recorded_at":"2022-06-09T13:01:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":43422,"utc_seconds":1654779771,"temperature_c":26.61,"humidity_pct":49.37,"tvoc_ppb":1228,"eco2_ppm":400,"raw_h2":12913,"raw_ethanol":19421,"pressure_hpa":938.703,"pm1_0":1.69,"pm2_5":1.75,"nc0_5":11.61,"nc1_0":1.81,"nc2_5":0.041,"cnt":18428,"fire_alarm":true,"recorded_at":"2022-06-09T13:02:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":43460,"utc_seconds":1654779809,"temperature_c":24.68,"humidity_pct":53.01,"tvoc_ppb":1201,"eco2_ppm":407,"raw_h2":12907,"raw_ethanol":19415,"pressure_hpa":938.743,"pm1_0":1.59,"pm2_5":1.65,"nc0_5":10.91,"nc1_0":1.701,"nc2_5":0.038,"cnt":18466,"fire_alarm":true,"recorded_at":"2022-06-09T13:03:29+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":43496,"utc_seconds":1654779845,"temperature_c":27.02,"humidity_pct":47.61,"tvoc_ppb":1191,"eco2_ppm":408,"raw_h2":12912,"raw_ethanol":19436,"pressure_hpa":938.742,"pm1_0":1.7,"pm2_5":1.77,"nc0_5":11.7,"nc1_0":1.825,"nc2_5":0.041,"cnt":18502,"fire_alarm":true,"recorded_at":"2022-06-09T13:04:05+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":44621,"utc_seconds":1654780970,"temperature_c":26.48,"humidity_pct":49.88,"tvoc_ppb":1145,"eco2_ppm":400,"raw_h2":12932,"raw_ethanol":19447,"pressure_hpa":938.698,"pm1_0":1.7,"pm2_5":1.77,"nc0_5":11.73,"nc1_0":1.829,"nc2_5":0.041,"cnt":19627,"fire_alarm":true,"recorded_at":"2022-06-09T13:22:50+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":44643,"utc_seconds":1654780992,"temperature_c":27.11,"humidity_pct":47.5,"tvoc_ppb":1212,"eco2_ppm":409,"raw_h2":12922,"raw_ethanol":19429,"pressure_hpa":938.7,"pm1_0":1.54,"pm2_5":1.6,"nc0_5":10.62,"nc1_0":1.656,"nc2_5":0.037,"cnt":19649,"fire_alarm":true,"recorded_at":"2022-06-09T13:23:12+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":44657,"utc_seconds":1654781006,"temperature_c":24.72,"humidity_pct":52.82,"tvoc_ppb":1231,"eco2_ppm":416,"raw_h2":12918,"raw_ethanol":19420,"pressure_hpa":938.715,"pm1_0":1.44,"pm2_5":1.5,"nc0_5":9.91,"nc1_0":1.545,"nc2_5":0.035,"cnt":19663,"fire_alarm":true,"recorded_at":"2022-06-09T13:23:26+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":45174,"utc_seconds":1654781523,"temperature_c":26.95,"humidity_pct":46.68,"tvoc_ppb":919,"eco2_ppm":400,"raw_h2":13026,"raw_ethanol":19502,"pressure_hpa":938.751,"pm1_0":2.09,"pm2_5":2.18,"nc0_5":14.41,"nc1_0":2.248,"nc2_5":0.051,"cnt":20180,"fire_alarm":true,"recorded_at":"2022-06-09T13:32:03+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":45220,"utc_seconds":1654781569,"temperature_c":26.4,"humidity_pct":50.06,"tvoc_ppb":1117,"eco2_ppm":400,"raw_h2":12958,"raw_ethanol":19449,"pressure_hpa":938.731,"pm1_0":2.09,"pm2_5":2.17,"nc0_5":14.39,"nc1_0":2.243,"nc2_5":0.051,"cnt":20226,"fire_alarm":true,"recorded_at":"2022-06-09T13:32:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":45260,"utc_seconds":1654781609,"temperature_c":24.55,"humidity_pct":53.53,"tvoc_ppb":1333,"eco2_ppm":425,"raw_h2":12924,"raw_ethanol":19397,"pressure_hpa":938.748,"pm1_0":1.45,"pm2_5":1.51,"nc0_5":10.01,"nc1_0":1.562,"nc2_5":0.035,"cnt":20266,"fire_alarm":true,"recorded_at":"2022-06-09T13:33:29+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":45731,"utc_seconds":1654782080,"temperature_c":24.92,"humidity_pct":50.6,"tvoc_ppb":1226,"eco2_ppm":400,"raw_h2":12971,"raw_ethanol":19423,"pressure_hpa":938.704,"pm1_0":2.4,"pm2_5":2.49,"nc0_5":16.49,"nc1_0":2.572,"nc2_5":0.058,"cnt":20737,"fire_alarm":true,"recorded_at":"2022-06-09T13:41:20+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":45824,"utc_seconds":1654782173,"temperature_c":26.63,"humidity_pct":48.65,"tvoc_ppb":1220,"eco2_ppm":400,"raw_h2":12958,"raw_ethanol":19419,"pressure_hpa":938.69,"pm1_0":2.33,"pm2_5":2.42,"nc0_5":16.01,"nc1_0":2.497,"nc2_5":0.056,"cnt":20830,"fire_alarm":true,"recorded_at":"2022-06-09T13:42:53+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":45858,"utc_seconds":1654782207,"temperature_c":24.64,"humidity_pct":52.65,"tvoc_ppb":1366,"eco2_ppm":417,"raw_h2":12947,"raw_ethanol":19383,"pressure_hpa":938.694,"pm1_0":1.93,"pm2_5":2.01,"nc0_5":13.31,"nc1_0":2.075,"nc2_5":0.047,"cnt":20864,"fire_alarm":true,"recorded_at":"2022-06-09T13:43:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":46318,"utc_seconds":1654782667,"temperature_c":26.95,"humidity_pct":47.66,"tvoc_ppb":1281,"eco2_ppm":400,"raw_h2":12958,"raw_ethanol":19402,"pressure_hpa":938.696,"pm1_0":1.98,"pm2_5":2.05,"nc0_5":13.61,"nc1_0":2.122,"nc2_5":0.048,"cnt":21324,"fire_alarm":true,"recorded_at":"2022-06-09T13:51:07+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":46658,"utc_seconds":1654783007,"temperature_c":26.22,"humidity_pct":51.56,"tvoc_ppb":1289,"eco2_ppm":400,"raw_h2":12971,"raw_ethanol":19402,"pressure_hpa":938.727,"pm1_0":2.41,"pm2_5":2.5,"nc0_5":16.56,"nc1_0":2.582,"nc2_5":0.058,"cnt":21664,"fire_alarm":true,"recorded_at":"2022-06-09T13:56:47+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":46681,"utc_seconds":1654783030,"temperature_c":26.99,"humidity_pct":47.61,"tvoc_ppb":1289,"eco2_ppm":400,"raw_h2":12959,"raw_ethanol":19395,"pressure_hpa":938.708,"pm1_0":2.47,"pm2_5":2.56,"nc0_5":16.98,"nc1_0":2.647,"nc2_5":0.06,"cnt":21687,"fire_alarm":true,"recorded_at":"2022-06-09T13:57:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":46836,"utc_seconds":1654783185,"temperature_c":25.89,"humidity_pct":53.37,"tvoc_ppb":1241,"eco2_ppm":400,"raw_h2":12978,"raw_ethanol":19411,"pressure_hpa":938.728,"pm1_0":1.45,"pm2_5":1.51,"nc0_5":10.01,"nc1_0":1.561,"nc2_5":0.035,"cnt":21842,"fire_alarm":true,"recorded_at":"2022-06-09T13:59:45+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":47053,"utc_seconds":1654783402,"temperature_c":24.85,"humidity_pct":52.19,"tvoc_ppb":1413,"eco2_ppm":421,"raw_h2":12960,"raw_ethanol":19384,"pressure_hpa":938.728,"pm1_0":1.86,"pm2_5":1.93,"nc0_5":12.79,"nc1_0":1.995,"nc2_5":0.045,"cnt":22059,"fire_alarm":true,"recorded_at":"2022-06-09T14:03:22+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":47924,"utc_seconds":1654784273,"temperature_c":26.57,"humidity_pct":49.22,"tvoc_ppb":1310,"eco2_ppm":400,"raw_h2":12984,"raw_ethanol":19413,"pressure_hpa":938.735,"pm1_0":1.74,"pm2_5":1.81,"nc0_5":11.96,"nc1_0":1.865,"nc2_5":0.042,"cnt":22930,"fire_alarm":true,"recorded_at":"2022-06-09T14:17:53+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":48015,"utc_seconds":1654784364,"temperature_c":24.72,"humidity_pct":52.73,"tvoc_ppb":1375,"eco2_ppm":413,"raw_h2":12970,"raw_ethanol":19396,"pressure_hpa":938.714,"pm1_0":1.95,"pm2_5":2.03,"nc0_5":13.44,"nc1_0":2.095,"nc2_5":0.047,"cnt":23021,"fire_alarm":true,"recorded_at":"2022-06-09T14:19:24+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":48038,"utc_seconds":1654784387,"temperature_c":26.22,"humidity_pct":51.71,"tvoc_ppb":1329,"eco2_ppm":400,"raw_h2":12977,"raw_ethanol":19405,"pressure_hpa":938.705,"pm1_0":1.85,"pm2_5":1.93,"nc0_5":12.76,"nc1_0":1.99,"nc2_5":0.045,"cnt":23044,"fire_alarm":true,"recorded_at":"2022-06-09T14:19:47+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":48084,"utc_seconds":1654784433,"temperature_c":24.47,"humidity_pct":53.73,"tvoc_ppb":1366,"eco2_ppm":408,"raw_h2":12967,"raw_ethanol":19383,"pressure_hpa":938.716,"pm1_0":2.13,"pm2_5":2.21,"nc0_5":14.63,"nc1_0":2.281,"nc2_5":0.052,"cnt":23090,"fire_alarm":true,"recorded_at":"2022-06-09T14:20:33+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":48271,"utc_seconds":1654784620,"temperature_c":24.37,"humidity_pct":54.11,"tvoc_ppb":1415,"eco2_ppm":413,"raw_h2":12967,"raw_ethanol":19378,"pressure_hpa":938.725,"pm1_0":1.85,"pm2_5":1.93,"nc0_5":12.75,"nc1_0":1.989,"nc2_5":0.045,"cnt":23277,"fire_alarm":true,"recorded_at":"2022-06-09T14:23:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":48574,"utc_seconds":1654784923,"temperature_c":24.31,"humidity_pct":54.29,"tvoc_ppb":1379,"eco2_ppm":406,"raw_h2":12971,"raw_ethanol":19385,"pressure_hpa":938.733,"pm1_0":1.81,"pm2_5":1.88,"nc0_5":12.44,"nc1_0":1.939,"nc2_5":0.044,"cnt":23580,"fire_alarm":true,"recorded_at":"2022-06-09T14:28:43+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":48611,"utc_seconds":1654784960,"temperature_c":24.89,"humidity_pct":51.73,"tvoc_ppb":1367,"eco2_ppm":407,"raw_h2":12967,"raw_ethanol":19374,"pressure_hpa":938.708,"pm1_0":1.53,"pm2_5":1.59,"nc0_5":10.52,"nc1_0":1.64,"nc2_5":0.037,"cnt":23617,"fire_alarm":true,"recorded_at":"2022-06-09T14:29:20+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":49022,"utc_seconds":1654785371,"temperature_c":27.01,"humidity_pct":47.69,"tvoc_ppb":1320,"eco2_ppm":400,"raw_h2":12998,"raw_ethanol":19404,"pressure_hpa":938.774,"pm1_0":1.56,"pm2_5":1.62,"nc0_5":10.76,"nc1_0":1.678,"nc2_5":0.038,"cnt":24028,"fire_alarm":true,"recorded_at":"2022-06-09T14:36:11+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":49265,"utc_seconds":1654785614,"temperature_c":25.81,"humidity_pct":47.13,"tvoc_ppb":1262,"eco2_ppm":400,"raw_h2":12998,"raw_ethanol":19405,"pressure_hpa":938.783,"pm1_0":2.25,"pm2_5":2.33,"nc0_5":15.46,"nc1_0":2.41,"nc2_5":0.054,"cnt":24271,"fire_alarm":true,"recorded_at":"2022-06-09T14:40:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":49385,"utc_seconds":1654785734,"temperature_c":25.84,"humidity_pct":47.53,"tvoc_ppb":1431,"eco2_ppm":435,"raw_h2":12976,"raw_ethanol":19374,"pressure_hpa":938.781,"pm1_0":2.11,"pm2_5":2.2,"nc0_5":14.54,"nc1_0":2.268,"nc2_5":0.051,"cnt":24391,"fire_alarm":true,"recorded_at":"2022-06-09T14:42:14+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":49633,"utc_seconds":1654785982,"temperature_c":24.76,"humidity_pct":52.37,"tvoc_ppb":1253,"eco2_ppm":401,"raw_h2":12990,"raw_ethanol":19409,"pressure_hpa":938.82,"pm1_0":2.32,"pm2_5":2.41,"nc0_5":15.98,"nc1_0":2.492,"nc2_5":0.056,"cnt":24639,"fire_alarm":true,"recorded_at":"2022-06-09T14:46:22+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":50414,"utc_seconds":1654903431,"temperature_c":39.62,"humidity_pct":16.02,"tvoc_ppb":0,"eco2_ppm":4742,"raw_h2":12336,"raw_ethanol":20106,"pressure_hpa":930.943,"pm1_0":8.09,"pm2_5":8.4,"nc0_5":55.66,"nc1_0":8.679,"nc2_5":0.196,"cnt":426,"fire_alarm":true,"recorded_at":"2022-06-10T23:23:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":50420,"utc_seconds":1654903437,"temperature_c":39.99,"humidity_pct":15.63,"tvoc_ppb":108,"eco2_ppm":9238,"raw_h2":12036,"raw_ethanol":19995,"pressure_hpa":930.967,"pm1_0":8.19,"pm2_5":8.51,"nc0_5":56.35,"nc1_0":8.787,"nc2_5":0.198,"cnt":432,"fire_alarm":true,"recorded_at":"2022-06-10T23:23:57+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":50750,"utc_seconds":1654903767,"temperature_c":37.02,"humidity_pct":15.75,"tvoc_ppb":0,"eco2_ppm":1364,"raw_h2":12881,"raw_ethanol":20742,"pressure_hpa":930.89,"pm1_0":1.09,"pm2_5":1.13,"nc0_5":7.5,"nc1_0":1.17,"nc2_5":0.026,"cnt":762,"fire_alarm":true,"recorded_at":"2022-06-10T23:29:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":51080,"utc_seconds":1654904097,"temperature_c":32.27,"humidity_pct":17.89,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13461,"raw_ethanol":21055,"pressure_hpa":930.907,"pm1_0":1.07,"pm2_5":1.11,"nc0_5":7.38,"nc1_0":1.151,"nc2_5":0.026,"cnt":1092,"fire_alarm":true,"recorded_at":"2022-06-10T23:34:57+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":51135,"utc_seconds":1654904152,"temperature_c":31.82,"humidity_pct":17.28,"tvoc_ppb":0,"eco2_ppm":746,"raw_h2":13137,"raw_ethanol":20961,"pressure_hpa":930.934,"pm1_0":1.08,"pm2_5":1.12,"nc0_5":7.4,"nc1_0":1.155,"nc2_5":0.026,"cnt":1147,"fire_alarm":true,"recorded_at":"2022-06-10T23:35:52+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":51430,"utc_seconds":1654712475,"temperature_c":29.74,"humidity_pct":39.43,"tvoc_ppb":33,"eco2_ppm":400,"raw_h2":12837,"raw_ethanol":20750,"pressure_hpa":937.589,"pm1_0":1.9,"pm2_5":1.98,"nc0_5":13.09,"nc1_0":2.041,"nc2_5":0.046,"cnt":288,"fire_alarm":false,"recorded_at":"2022-06-08T18:21:15+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":51620,"utc_seconds":1654712665,"temperature_c":27.18,"humidity_pct":43.8,"tvoc_ppb":67,"eco2_ppm":410,"raw_h2":12827,"raw_ethanol":20700,"pressure_hpa":937.533,"pm1_0":2.18,"pm2_5":2.27,"nc0_5":15.02,"nc1_0":2.342,"nc2_5":0.053,"cnt":478,"fire_alarm":false,"recorded_at":"2022-06-08T18:24:25+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":51780,"utc_seconds":1654712825,"temperature_c":29.2,"humidity_pct":40.76,"tvoc_ppb":63,"eco2_ppm":419,"raw_h2":12809,"raw_ethanol":20701,"pressure_hpa":937.501,"pm1_0":2.15,"pm2_5":2.24,"nc0_5":14.82,"nc1_0":2.311,"nc2_5":0.052,"cnt":638,"fire_alarm":false,"recorded_at":"2022-06-08T18:27:05+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52135,"utc_seconds":1654713180,"temperature_c":28.79,"humidity_pct":41.73,"tvoc_ppb":72,"eco2_ppm":400,"raw_h2":12795,"raw_ethanol":20682,"pressure_hpa":937.535,"pm1_0":2.22,"pm2_5":2.31,"nc0_5":15.28,"nc1_0":2.383,"nc2_5":0.054,"cnt":993,"fire_alarm":false,"recorded_at":"2022-06-08T18:33:00+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52650,"utc_seconds":1654713695,"temperature_c":26.13,"humidity_pct":46.92,"tvoc_ppb":96,"eco2_ppm":400,"raw_h2":12782,"raw_ethanol":20630,"pressure_hpa":937.423,"pm1_0":1.81,"pm2_5":1.88,"nc0_5":12.46,"nc1_0":1.944,"nc2_5":0.044,"cnt":1508,"fire_alarm":false,"recorded_at":"2022-06-08T18:41:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52733,"utc_seconds":1654713778,"temperature_c":28.45,"humidity_pct":42.61,"tvoc_ppb":80,"eco2_ppm":400,"raw_h2":12791,"raw_ethanol":20641,"pressure_hpa":937.408,"pm1_0":1.73,"pm2_5":1.79,"nc0_5":11.87,"nc1_0":1.852,"nc2_5":0.042,"cnt":1591,"fire_alarm":false,"recorded_at":"2022-06-08T18:42:58+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52810,"utc_seconds":1654713855,"temperature_c":28.93,"humidity_pct":41.12,"tvoc_ppb":62,"eco2_ppm":400,"raw_h2":12798,"raw_ethanol":20655,"pressure_hpa":937.39,"pm1_0":1.72,"pm2_5":1.78,"nc0_5":11.81,"nc1_0":1.842,"nc2_5":0.042,"cnt":1668,"fire_alarm":false,"recorded_at":"2022-06-08T18:44:15+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52835,"utc_seconds":1654713880,"temperature_c":25.96,"humidity_pct":47.41,"tvoc_ppb":122,"eco2_ppm":407,"raw_h2":12777,"raw_ethanol":20605,"pressure_hpa":937.379,"pm1_0":1.98,"pm2_5":2.06,"nc0_5":13.66,"nc1_0":2.13,"nc2_5":0.048,"cnt":1693,"fire_alarm":false,"recorded_at":"2022-06-08T18:44:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52906,"utc_seconds":1654713951,"temperature_c":27.64,"humidity_pct":46.91,"tvoc_ppb":73,"eco2_ppm":400,"raw_h2":12801,"raw_ethanol":20655,"pressure_hpa":937.377,"pm1_0":1.88,"pm2_5":1.95,"nc0_5":12.95,"nc1_0":2.02,"nc2_5":0.046,"cnt":1764,"fire_alarm":false,"recorded_at":"2022-06-08T18:45:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52964,"utc_seconds":1654714009,"temperature_c":25.74,"humidity_pct":48.04,"tvoc_ppb":148,"eco2_ppm":411,"raw_h2":12776,"raw_ethanol":20577,"pressure_hpa":937.419,"pm1_0":2.17,"pm2_5":2.26,"nc0_5":14.96,"nc1_0":2.333,"nc2_5":0.053,"cnt":1822,"fire_alarm":false,"recorded_at":"2022-06-08T18:46:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":52992,"utc_seconds":1654714037,"temperature_c":28.9,"humidity_pct":41.19,"tvoc_ppb":151,"eco2_ppm":400,"raw_h2":12782,"raw_ethanol":20569,"pressure_hpa":937.433,"pm1_0":1.62,"pm2_5":1.69,"nc0_5":11.17,"nc1_0":1.742,"nc2_5":0.039,"cnt":1850,"fire_alarm":false,"recorded_at":"2022-06-08T18:47:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":53112,"utc_seconds":1654714157,"temperature_c":28.86,"humidity_pct":41.24,"tvoc_ppb":129,"eco2_ppm":404,"raw_h2":12788,"raw_ethanol":20596,"pressure_hpa":937.352,"pm1_0":2.03,"pm2_5":2.11,"nc0_5":13.98,"nc1_0":2.18,"nc2_5":0.049,"cnt":1970,"fire_alarm":false,"recorded_at":"2022-06-08T18:49:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":53231,"utc_seconds":1654714276,"temperature_c":28.84,"humidity_pct":41.34,"tvoc_ppb":125,"eco2_ppm":404,"raw_h2":12786,"raw_ethanol":20600,"pressure_hpa":937.34,"pm1_0":1.88,"pm2_5":1.95,"nc0_5":12.93,"nc1_0":2.017,"nc2_5":0.046,"cnt":2089,"fire_alarm":false,"recorded_at":"2022-06-08T18:51:16+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":53251,"utc_seconds":1654714296,"temperature_c":25.98,"humidity_pct":47.42,"tvoc_ppb":117,"eco2_ppm":413,"raw_h2":12781,"raw_ethanol":20604,"pressure_hpa":937.357,"pm1_0":1.61,"pm2_5":1.67,"nc0_5":11.05,"nc1_0":1.723,"nc2_5":0.039,"cnt":2109,"fire_alarm":false,"recorded_at":"2022-06-08T18:51:36+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":53652,"utc_seconds":1654714697,"temperature_c":28.86,"humidity_pct":41.98,"tvoc_ppb":161,"eco2_ppm":429,"raw_h2":12774,"raw_ethanol":20555,"pressure_hpa":937.321,"pm1_0":1.81,"pm2_5":1.88,"nc0_5":12.46,"nc1_0":1.943,"nc2_5":0.044,"cnt":2510,"fire_alarm":false,"recorded_at":"2022-06-08T18:58:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":53692,"utc_seconds":1654714737,"temperature_c":28.27,"humidity_pct":44.0,"tvoc_ppb":152,"eco2_ppm":419,"raw_h2":12776,"raw_ethanol":20566,"pressure_hpa":937.324,"pm1_0":1.83,"pm2_5":1.9,"nc0_5":12.59,"nc1_0":1.963,"nc2_5":0.044,"cnt":2550,"fire_alarm":false,"recorded_at":"2022-06-08T18:58:57+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":54046,"utc_seconds":1654715091,"temperature_c":27.6,"humidity_pct":48.06,"tvoc_ppb":156,"eco2_ppm":400,"raw_h2":12786,"raw_ethanol":20562,"pressure_hpa":937.409,"pm1_0":1.74,"pm2_5":1.8,"nc0_5":11.95,"nc1_0":1.864,"nc2_5":0.042,"cnt":2904,"fire_alarm":false,"recorded_at":"2022-06-08T19:04:51+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":54150,"utc_seconds":1654715195,"temperature_c":25.99,"humidity_pct":48.33,"tvoc_ppb":181,"eco2_ppm":431,"raw_h2":12774,"raw_ethanol":20549,"pressure_hpa":937.341,"pm1_0":1.9,"pm2_5":1.97,"nc0_5":13.07,"nc1_0":2.038,"nc2_5":0.046,"cnt":3008,"fire_alarm":false,"recorded_at":"2022-06-08T19:06:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":54354,"utc_seconds":1654715399,"temperature_c":28.38,"humidity_pct":43.69,"tvoc_ppb":190,"eco2_ppm":412,"raw_h2":12776,"raw_ethanol":20518,"pressure_hpa":937.335,"pm1_0":1.73,"pm2_5":1.79,"nc0_5":11.88,"nc1_0":1.853,"nc2_5":0.042,"cnt":3212,"fire_alarm":false,"recorded_at":"2022-06-08T19:09:59+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":54442,"utc_seconds":1654715487,"temperature_c":26.39,"humidity_pct":47.0,"tvoc_ppb":205,"eco2_ppm":441,"raw_h2":12763,"raw_ethanol":20513,"pressure_hpa":937.411,"pm1_0":1.66,"pm2_5":1.72,"nc0_5":11.41,"nc1_0":1.78,"nc2_5":0.04,"cnt":3300,"fire_alarm":false,"recorded_at":"2022-06-08T19:11:27+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":54736,"utc_seconds":1654715781,"temperature_c":26.47,"humidity_pct":39.36,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13235,"raw_ethanol":21226,"pressure_hpa":937.46,"pm1_0":1.8,"pm2_5":1.87,"nc0_5":12.36,"nc1_0":1.927,"nc2_5":0.044,"cnt":3594,"fire_alarm":false,"recorded_at":"2022-06-08T19:16:21+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":55368,"utc_seconds":1654716413,"temperature_c":41.69,"humidity_pct":35.18,"tvoc_ppb":10136,"eco2_ppm":400,"raw_h2":12858,"raw_ethanol":19296,"pressure_hpa":936.86,"pm1_0":0.44,"pm2_5":0.46,"nc0_5":3.04,"nc1_0":0.473,"nc2_5":0.011,"cnt":4226,"fire_alarm":false,"recorded_at":"2022-06-08T19:26:53+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":55564,"utc_seconds":1654716609,"temperature_c":49.95,"humidity_pct":24.31,"tvoc_ppb":21060,"eco2_ppm":653,"raw_h2":12679,"raw_ethanol":18931,"pressure_hpa":936.897,"pm1_0":0.73,"pm2_5":0.75,"nc0_5":4.99,"nc1_0":0.778,"nc2_5":0.018,"cnt":4422,"fire_alarm":false,"recorded_at":"2022-06-08T19:30:09+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":55657,"utc_seconds":1654716702,"temperature_c":51.4,"humidity_pct":23.49,"tvoc_ppb":31413,"eco2_ppm":711,"raw_h2":12651,"raw_ethanol":18766,"pressure_hpa":936.919,"pm1_0":0.62,"pm2_5":0.64,"nc0_5":4.24,"nc1_0":0.661,"nc2_5":0.015,"cnt":4515,"fire_alarm":false,"recorded_at":"2022-06-08T19:31:42+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":55898,"utc_seconds":1654716943,"temperature_c":42.14,"humidity_pct":19.57,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13585,"raw_ethanol":21185,"pressure_hpa":936.804,"pm1_0":0.91,"pm2_5":0.95,"nc0_5":6.29,"nc1_0":0.981,"nc2_5":0.022,"cnt":4756,"fire_alarm":false,"recorded_at":"2022-06-08T19:35:43+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":56004,"utc_seconds":1654717049,"temperature_c":43.74,"humidity_pct":32.63,"tvoc_ppb":60000,"eco2_ppm":1420,"raw_h2":12279,"raw_ethanol":18153,"pressure_hpa":936.862,"pm1_0":35.49,"pm2_5":36.88,"nc0_5":244.28,"nc1_0":38.092,"nc2_5":0.86,"cnt":4862,"fire_alarm":false,"recorded_at":"2022-06-08T19:37:29+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":56073,"utc_seconds":1654717118,"temperature_c":46.7,"humidity_pct":27.66,"tvoc_ppb":60000,"eco2_ppm":8948,"raw_h2":11424,"raw_ethanol":16745,"pressure_hpa":936.867,"pm1_0":2962.12,"pm2_5":3077.53,"nc0_5":20387.05,"nc1_0":3179.117,"nc2_5":71.805,"cnt":4931,"fire_alarm":false,"recorded_at":"2022-06-08T19:38:38+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":56551,"utc_seconds":1654717596,"temperature_c":54.69,"humidity_pct":12.49,"tvoc_ppb":9636,"eco2_ppm":400,"raw_h2":13251,"raw_ethanol":19398,"pressure_hpa":936.698,"pm1_0":0.84,"pm2_5":0.88,"nc0_5":5.81,"nc1_0":0.907,"nc2_5":0.02,"cnt":5409,"fire_alarm":false,"recorded_at":"2022-06-08T19:46:36+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":56850,"utc_seconds":1654717895,"temperature_c":44.39,"humidity_pct":17.02,"tvoc_ppb":559,"eco2_ppm":400,"raw_h2":13771,"raw_ethanol":20631,"pressure_hpa":936.69,"pm1_0":0.81,"pm2_5":0.84,"nc0_5":5.55,"nc1_0":0.865,"nc2_5":0.02,"cnt":5708,"fire_alarm":false,"recorded_at":"2022-06-08T19:51:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":56885,"utc_seconds":1654717930,"temperature_c":44.28,"humidity_pct":16.52,"tvoc_ppb":643,"eco2_ppm":400,"raw_h2":13696,"raw_ethanol":20543,"pressure_hpa":936.676,"pm1_0":0.57,"pm2_5":0.59,"nc0_5":3.9,"nc1_0":0.607,"nc2_5":0.014,"cnt":5743,"fire_alarm":false,"recorded_at":"2022-06-08T19:52:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":57318,"utc_seconds":1655124740,"temperature_c":23.542,"humidity_pct":45.86,"tvoc_ppb":62,"eco2_ppm":400,"raw_h2":12833,"raw_ethanol":20713,"pressure_hpa":937.503,"pm1_0":1.68,"pm2_5":1.74,"nc0_5":11.54,"nc1_0":1.8,"nc2_5":0.041,"cnt":432,"fire_alarm":false,"recorded_at":"2022-06-13T12:52:20+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58013,"utc_seconds":1655125435,"temperature_c":-7.903,"humidity_pct":40.79,"tvoc_ppb":108,"eco2_ppm":400,"raw_h2":12786,"raw_ethanol":20636,"pressure_hpa":937.49,"pm1_0":1.59,"pm2_5":1.65,"nc0_5":10.93,"nc1_0":1.704,"nc2_5":0.038,"cnt":1127,"fire_alarm":false,"recorded_at":"2022-06-13T13:03:55+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58048,"utc_seconds":1655125470,"temperature_c":-8.542,"humidity_pct":47.59,"tvoc_ppb":122,"eco2_ppm":402,"raw_h2":12778,"raw_ethanol":20619,"pressure_hpa":937.462,"pm1_0":1.93,"pm2_5":2.0,"nc0_5":13.26,"nc1_0":2.068,"nc2_5":0.047,"cnt":1162,"fire_alarm":false,"recorded_at":"2022-06-13T13:04:30+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58260,"utc_seconds":1655125682,"temperature_c":-8.648,"humidity_pct":41.77,"tvoc_ppb":107,"eco2_ppm":424,"raw_h2":12771,"raw_ethanol":20619,"pressure_hpa":937.454,"pm1_0":1.84,"pm2_5":1.91,"nc0_5":12.68,"nc1_0":1.977,"nc2_5":0.045,"cnt":1374,"fire_alarm":false,"recorded_at":"2022-06-13T13:08:02+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58420,"utc_seconds":1655125842,"temperature_c":-6.413,"humidity_pct":41.97,"tvoc_ppb":79,"eco2_ppm":400,"raw_h2":12791,"raw_ethanol":20641,"pressure_hpa":937.401,"pm1_0":2.13,"pm2_5":2.21,"nc0_5":14.65,"nc1_0":2.284,"nc2_5":0.052,"cnt":1534,"fire_alarm":false,"recorded_at":"2022-06-13T13:10:42+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58563,"utc_seconds":1655125985,"temperature_c":-4.544,"humidity_pct":44.51,"tvoc_ppb":85,"eco2_ppm":415,"raw_h2":12784,"raw_ethanol":20635,"pressure_hpa":937.394,"pm1_0":1.89,"pm2_5":1.96,"nc0_5":12.99,"nc1_0":2.025,"nc2_5":0.046,"cnt":1677,"fire_alarm":false,"recorded_at":"2022-06-13T13:13:05+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58650,"utc_seconds":1655126072,"temperature_c":-4.09,"humidity_pct":46.91,"tvoc_ppb":73,"eco2_ppm":400,"raw_h2":12801,"raw_ethanol":20655,"pressure_hpa":937.377,"pm1_0":1.88,"pm2_5":1.95,"nc0_5":12.95,"nc1_0":2.02,"nc2_5":0.046,"cnt":1764,"fire_alarm":false,"recorded_at":"2022-06-13T13:14:32+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":58882,"utc_seconds":1655126304,"temperature_c":-4.992,"humidity_pct":47.92,"tvoc_ppb":124,"eco2_ppm":415,"raw_h2":12782,"raw_ethanol":20594,"pressure_hpa":937.336,"pm1_0":1.96,"pm2_5":2.04,"nc0_5":13.5,"nc1_0":2.105,"nc2_5":0.048,"cnt":1996,"fire_alarm":false,"recorded_at":"2022-06-13T13:18:24+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":59162,"utc_seconds":1655126584,"temperature_c":-3.419,"humidity_pct":44.19,"tvoc_ppb":133,"eco2_ppm":418,"raw_h2":12781,"raw_ethanol":20577,"pressure_hpa":937.324,"pm1_0":1.81,"pm2_5":1.88,"nc0_5":12.45,"nc1_0":1.941,"nc2_5":0.044,"cnt":2276,"fire_alarm":false,"recorded_at":"2022-06-13T13:23:04+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":59435,"utc_seconds":1655126857,"temperature_c":7.347,"humidity_pct":44.27,"tvoc_ppb":168,"eco2_ppm":427,"raw_h2":12776,"raw_ethanol":20559,"pressure_hpa":937.331,"pm1_0":1.85,"pm2_5":1.92,"nc0_5":12.72,"nc1_0":1.983,"nc2_5":0.045,"cnt":2549,"fire_alarm":false,"recorded_at":"2022-06-13T13:27:37+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":59517,"utc_seconds":1655126939,"temperature_c":11.323,"humidity_pct":42.03,"tvoc_ppb":152,"eco2_ppm":424,"raw_h2":12777,"raw_ethanol":20562,"pressure_hpa":937.365,"pm1_0":2.08,"pm2_5":2.16,"nc0_5":14.28,"nc1_0":2.227,"nc2_5":0.05,"cnt":2631,"fire_alarm":false,"recorded_at":"2022-06-13T13:28:59+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":59930,"utc_seconds":1655127352,"temperature_c":20.533,"humidity_pct":42.51,"tvoc_ppb":186,"eco2_ppm":428,"raw_h2":12768,"raw_ethanol":20529,"pressure_hpa":937.341,"pm1_0":1.79,"pm2_5":1.86,"nc0_5":12.33,"nc1_0":1.922,"nc2_5":0.043,"cnt":3044,"fire_alarm":false,"recorded_at":"2022-06-13T13:35:52+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":59937,"utc_seconds":1655127359,"temperature_c":20.468,"humidity_pct":42.2,"tvoc_ppb":181,"eco2_ppm":423,"raw_h2":12774,"raw_ethanol":20538,"pressure_hpa":937.348,"pm1_0":1.56,"pm2_5":1.62,"nc0_5":10.73,"nc1_0":1.673,"nc2_5":0.038,"cnt":3051,"fire_alarm":false,"recorded_at":"2022-06-13T13:35:59+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":60261,"utc_seconds":1655127683,"temperature_c":14.391,"humidity_pct":49.31,"tvoc_ppb":232,"eco2_ppm":450,"raw_h2":12763,"raw_ethanol":20485,"pressure_hpa":937.452,"pm1_0":1.74,"pm2_5":1.81,"nc0_5":11.98,"nc1_0":1.868,"nc2_5":0.042,"cnt":3375,"fire_alarm":false,"recorded_at":"2022-06-13T13:41:23+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":60652,"utc_seconds":1655128074,"temperature_c":12.612,"humidity_pct":38.33,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13444,"raw_ethanol":21295,"pressure_hpa":937.395,"pm1_0":1.79,"pm2_5":1.86,"nc0_5":12.31,"nc1_0":1.92,"nc2_5":0.043,"cnt":3766,"fire_alarm":false,"recorded_at":"2022-06-13T13:47:54+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":60931,"utc_seconds":1655128353,"temperature_c":16.584,"humidity_pct":59.68,"tvoc_ppb":2534,"eco2_ppm":400,"raw_h2":13354,"raw_ethanol":19985,"pressure_hpa":936.928,"pm1_0":0.95,"pm2_5":0.99,"nc0_5":6.55,"nc1_0":1.022,"nc2_5":0.023,"cnt":4045,"fire_alarm":false,"recorded_at":"2022-06-13T13:52:33+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61394,"utc_seconds":1655128816,"temperature_c":19.345,"humidity_pct":22.67,"tvoc_ppb":29991,"eco2_ppm":713,"raw_h2":12650,"raw_ethanol":18790,"pressure_hpa":936.938,"pm1_0":0.88,"pm2_5":0.92,"nc0_5":6.08,"nc1_0":0.948,"nc2_5":0.021,"cnt":4508,"fire_alarm":false,"recorded_at":"2022-06-13T14:00:16+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61397,"utc_seconds":1655128819,"temperature_c":19.278,"humidity_pct":22.89,"tvoc_ppb":30331,"eco2_ppm":696,"raw_h2":12654,"raw_ethanol":18788,"pressure_hpa":936.931,"pm1_0":0.76,"pm2_5":0.79,"nc0_5":5.21,"nc1_0":0.813,"nc2_5":0.018,"cnt":4511,"fire_alarm":false,"recorded_at":"2022-06-13T14:00:19+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61487,"utc_seconds":1655128909,"temperature_c":16.416,"humidity_pct":18.13,"tvoc_ppb":54715,"eco2_ppm":1016,"raw_h2":12494,"raw_ethanol":18393,"pressure_hpa":936.913,"pm1_0":0.85,"pm2_5":0.89,"nc0_5":5.87,"nc1_0":0.916,"nc2_5":0.021,"cnt":4601,"fire_alarm":false,"recorded_at":"2022-06-13T14:01:49+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61658,"utc_seconds":1655129080,"temperature_c":6.746,"humidity_pct":18.38,"tvoc_ppb":0,"eco2_ppm":400,"raw_h2":13607,"raw_ethanol":21255,"pressure_hpa":936.807,"pm1_0":1.17,"pm2_5":1.21,"nc0_5":8.03,"nc1_0":1.253,"nc2_5":0.028,"cnt":4772,"fire_alarm":false,"recorded_at":"2022-06-13T14:04:40+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61721,"utc_seconds":1655129143,"temperature_c":2.335,"humidity_pct":27.62,"tvoc_ppb":16640,"eco2_ppm":400,"raw_h2":12861,"raw_ethanol":19039,"pressure_hpa":936.85,"pm1_0":24.38,"pm2_5":25.33,"nc0_5":167.81,"nc1_0":26.168,"nc2_5":0.591,"cnt":4835,"fire_alarm":false,"recorded_at":"2022-06-13T14:05:43+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61748,"utc_seconds":1655129170,"temperature_c":0.387,"humidity_pct":32.63,"tvoc_ppb":60000,"eco2_ppm":1420,"raw_h2":12279,"raw_ethanol":18153,"pressure_hpa":936.862,"pm1_0":35.49,"pm2_5":36.88,"nc0_5":244.28,"nc1_0":38.092,"nc2_5":0.86,"cnt":4862,"fire_alarm":false,"recorded_at":"2022-06-13T14:06:10+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61755,"utc_seconds":1655129177,"temperature_c":-0.12,"humidity_pct":31.24,"tvoc_ppb":60000,"eco2_ppm":1660,"raw_h2":12191,"raw_ethanol":18012,"pressure_hpa":936.878,"pm1_0":225.18,"pm2_5":233.95,"nc0_5":1549.82,"nc1_0":241.676,"nc2_5":5.458,"cnt":4869,"fire_alarm":false,"recorded_at":"2022-06-13T14:06:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":61935,"utc_seconds":1655129357,"temperature_c":-12.598,"humidity_pct":24.55,"tvoc_ppb":60000,"eco2_ppm":13183,"raw_h2":11209,"raw_ethanol":16086,"pressure_hpa":936.846,"pm1_0":1074.1,"pm2_5":1338.02,"nc0_5":6834.65,"nc1_0":1418.372,"nc2_5":241.593,"cnt":5049,"fire_alarm":false,"recorded_at":"2022-06-13T14:09:17+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":62133,"utc_seconds":1655129555,"temperature_c":-21.292,"humidity_pct":11.57,"tvoc_ppb":60000,"eco2_ppm":8887,"raw_h2":11420,"raw_ethanol":16856,"pressure_hpa":936.755,"pm1_0":1124.89,"pm2_5":1312.62,"nc0_5":7380.66,"nc1_0":1379.394,"nc2_5":166.943,"cnt":5247,"fire_alarm":false,"recorded_at":"2022-06-13T14:12:35+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":62263,"utc_seconds":1655129685,"temperature_c":-21.31,"humidity_pct":11.06,"tvoc_ppb":9567,"eco2_ppm":400,"raw_h2":13227,"raw_ethanol":19402,"pressure_hpa":936.715,"pm1_0":25.6,"pm2_5":26.6,"nc0_5":176.2,"nc1_0":27.477,"nc2_5":0.621,"cnt":5377,"fire_alarm":false,"recorded_at":"2022-06-13T14:14:45+00:00","dataset_id":"smoke-detection-iot-300-v1"},{"source_row":62500,"utc_seconds":1655129922,"temperature_c":-3.324,"humidity_pct":13.7,"tvoc_ppb":1373,"eco2_ppm":400,"raw_h2":13663,"raw_ethanol":20323,"pressure_hpa":936.685,"pm1_0":0.71,"pm2_5":0.74,"nc0_5":4.91,"nc1_0":0.766,"nc2_5":0.017,"cnt":5614,"fire_alarm":false,"recorded_at":"2022-06-13T14:18:42+00:00","dataset_id":"smoke-detection-iot-300-v1"}]'
assert hashlib.sha256(payload.encode()).hexdigest() == 'c3d4216487bd938eda7ca27602f15fc65ec7b395f2203b5cd719d20a43615a78'
df = pd.DataFrame(json.loads(payload))
assert len(df) == 300 and not df.duplicated(['dataset_id','source_row']).any()
assert df['fire_alarm'].value_counts().to_dict() == {True:214,False:86}
df.to_csv(OUT/'lecturas_300.csv', index=False)
print('300 lecturas verificadas: 214 con alarma y 86 sin alarma.')
display(df.head())


## 2. Variables y separación temporal
Se excluyen la etiqueta, el identificador de origen, el contador CNT, el tiempo y el identificador del conjunto de datos. No deben usarse como pistas artificiales de la clase.
Las 12 características conservan las unidades de la fuente. No se inventan porcentajes de humo ni equivalencias con sensores MQ-2. No existe separación física por nodos: los seis nodos del panel son agrupaciones virtuales.


In [ ]:
FEATURES = ['temperature_c','humidity_pct','tvoc_ppb','eco2_ppm','raw_h2',
            'raw_ethanol','pressure_hpa','pm1_0','pm2_5','nc0_5','nc1_0','nc2_5']
assert not {'source_row','cnt','utc_seconds','recorded_at','fire_alarm','dataset_id'} & set(FEATURES)
df = df.sort_values(['utc_seconds','source_row']).reset_index(drop=True)
X = df[FEATURES].apply(pd.to_numeric, errors='raise')
assert np.isfinite(X.to_numpy()).all(), 'La muestra original contiene valores no finitos'
y = df['fire_alarm'].astype(int)
train_ids, test_ids = np.arange(240), np.arange(240,300)
X_train, X_test = X.iloc[train_ids], X.iloc[test_ids]
y_train, y_test = y.iloc[train_ids], y.iloc[test_ids]
assert df.iloc[train_ids]['utc_seconds'].max() < df.iloc[test_ids]['utc_seconds'].min()
assert y_train.nunique() == y_test.nunique() == 2
assert set(df.iloc[train_ids].source_row).isdisjoint(df.iloc[test_ids].source_row)
print('Entrenamiento:',len(X_train), y_train.value_counts().to_dict())
print('Prueba:',len(X_test), y_test.value_counts().to_dict())


## 3. Entrenar sin utilizar la prueba
El preprocesamiento se ajusta solo dentro del entrenamiento mediante Pipeline. La configuración del modelo es fija; no hay selección del mejor modelo usando la prueba.

In [ ]:
model = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('forest',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=3,
                                     class_weight='balanced',random_state=20260923,n_jobs=-1))
])
model.fit(X_train,y_train)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
pred = model.predict(X_test)
score = model.predict_proba(X_test)[:,list(model.classes_).index(1)]
def metrics(actual,predicted):
    return {'accuracy':float(accuracy_score(actual,predicted)),
            'balanced_accuracy':float(balanced_accuracy_score(actual,predicted)),
            'precision_alarm':float(precision_score(actual,predicted,zero_division=0)),
            'recall_alarm':float(recall_score(actual,predicted,zero_division=0)),
            'f1_alarm':float(f1_score(actual,predicted,zero_division=0))}
report = {'dataset_sha256':hashlib.sha256(payload.encode()).hexdigest(),
          'train_rows':240,'test_rows':60,'split':'chronological_80_20',
          'features':FEATURES,'random_forest':metrics(y_test,pred),
          'baseline':metrics(y_test,baseline.predict(X_test)),
          'roc_auc':float(roc_auc_score(y_test,score)),
          'confusion_matrix':confusion_matrix(y_test,pred,labels=[0,1]).tolist(),
          'sklearn_version':sklearn.__version__,
          'notice':'Experimental historical classification; not calibrated wildfire probability.'}
display(pd.DataFrame({'Random Forest':report['random_forest'],'Referencia mayoritaria':report['baseline']}))
print('ROC AUC:',report['roc_auc'])
print('No se han reajustado parámetros ni entrenado sobre las 60 filas de prueba.')


## 4. Evaluar errores y conservar las predicciones de prueba
Filas de la matriz: etiqueta real. Columnas: predicción. Un falso negativo es una lectura con etiqueta positiva clasificada como negativa. No interpretar la exactitud aislada como garantía de seguridad.
El score es la salida no calibrada del clasificador; no es la probabilidad comprobada de incendio en Satipo.


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=[0,1],display_labels=['Sin alarma','Con alarma'],cmap='Greens')
plt.title('Prueba temporal: 60 lecturas no usadas al entrenar')
plt.tight_layout();plt.savefig(OUT/'matriz_confusion.png',dpi=180);plt.show()
predictions = df.iloc[test_ids][['dataset_id','source_row','recorded_at','fire_alarm']].copy()
predictions['predicted_fire_alarm'] = pred.astype(bool)
predictions['model_score_alarm_uncalibrated'] = score
predictions['correct'] = predictions.fire_alarm == predictions.predicted_fire_alarm
predictions.to_csv(OUT/'predicciones_prueba_60.csv',index=False)
(OUT/'evaluacion.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
display(predictions.head(10))
print('Errores en prueba:',int((~predictions.correct).sum()))


## 5. Predecir nuevas mediciones con el modelo entrenado
La función exige los mismos 12 campos y valores numéricos finitos. Los valores fuera del rango observado se marcan como fuera de distribución, sin garantizar extrapolación.
El ejemplo siguiente usa una fila de prueba ya evaluada: es una demostración de inferencia, no una observación nueva ni una validación adicional. Sustituye `nueva_medicion` por mediciones compatibles para un ensayo propio.


In [ ]:
def predecir_medicion(medicion):
    missing = set(FEATURES) - set(medicion)
    if missing: raise ValueError('Faltan campos: ' + ', '.join(sorted(missing)))
    values = pd.DataFrame([{key:medicion[key] for key in FEATURES}]).apply(pd.to_numeric,errors='raise')
    if not np.isfinite(values.to_numpy()).all(): raise ValueError('Se requieren valores finitos')
    if not 0 <= values.humidity_pct.iloc[0] <= 100: raise ValueError('Humedad fuera de 0–100 %')
    outside = [key for key in FEATURES if not X_train[key].min() <= values[key].iloc[0] <= X_train[key].max()]
    return {'predicted_fire_alarm':bool(model.predict(values)[0]),
            'model_score_alarm_uncalibrated':float(model.predict_proba(values)[0,list(model.classes_).index(1)]),
            'outside_training_range':outside,'model_scope':'experimental_historical_classifier'}
nueva_medicion = X_test.iloc[0].to_dict()  # Ejemplo histórico; reemplazar para otra inferencia.
display(predecir_medicion(nueva_medicion))
try:
    predecir_medicion({})
    raise AssertionError('La validación no rechazó campos ausentes')
except ValueError:
    print('Validación de entrada comprobada.')


## 6. Exportar el modelo y resultados
Se exporta el modelo evaluado, entrenado únicamente con 240 filas. No se reentrena automáticamente con las 300: eso eliminaría el conjunto reservado para comprobar este artefacto.
El archivo joblib requiere versiones compatibles de Python/scikit-learn y solo debe cargarse desde una fuente confiable. No es un archivo ejecutable por el navegador ni sustituye los umbrales JSON del panel. Integrarlo al backend requiere un adaptador de inferencia y validación adicional.


In [ ]:
artifact = {'model':model,'features':FEATURES,'evaluation':report,
            'training_min':X_train.min().to_dict(),'training_max':X_train.max().to_dict()}
joblib.dump(artifact,OUT/'modelo_fire_alarm_experimental.joblib')
(OUT/'README.txt').write_text('Modelo experimental. 240 filas de entrenamiento, 60 de prueba.\n'
    'No usar como alarma operacional. Las probabilidades no están calibradas.\n'
    'Ver evaluacion.json y versiones antes de reproducir.\n',encoding='utf-8')
archive = Path('alerta_satipo_modelo_experimental.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for file in OUT.iterdir(): z.write(file,arcname=file.name)
print('Exportado:',archive.resolve())
try:
    from google.colab import files
except ImportError:
    print('Ejecución local: el ZIP está disponible en el directorio de trabajo.')
else:
    files.download(str(archive))


## Fuentes y limitaciones
- Fuente de los datos: https://raw.githubusercontent.com/HamzaMa96/Smoke-Detection-IOT/main/smoke_detection_iot.csv
- Prevención de fuga de datos: https://scikit-learn.org/1.8/common_pitfalls.html
- Uso de notebooks en Colab: https://research.google.com/colaboratory/faq.html?hl=es

La separación temporal reduce la mezcla de pasado y futuro, pero no elimina toda dependencia entre mediciones. Solo hay 60 filas de prueba y no se conoce aquí la sesión física de cada lectura. Antes de emplear el modelo para alertas: datos independientes de campo, evaluación por sesión/sensor, control de falsas alarmas, calibración y seguimiento de cambios de distribución.
